# Random Forest Peak Risk Classifier

## Ontario Electricity Peak-Risk Forecasting

This notebook develops a Random Forest Classifier to identify high-risk electricity-demand periods using forecast outputs generated by the forecasting models developed by the project team.

The Peak Risk Classifier is designed as a model-agnostic downstream component. It does not depend on the internal architecture of any individual forecasting model. Instead, each forecasting model provides its demand predictions through a standardized forecast interface.

## Forecasting Models

The project evaluates five electricity-demand forecasting approaches:

| Forecasting Model | Category | Role |
|---|---|---|
| Seasonal Naïve | Baseline | Uses historical seasonal demand as a simple forecasting benchmark |
| SARIMAX | Statistical | Models temporal dependence and seasonality with calendar and weather information |
| XGBoost Regressor | Machine Learning | Models nonlinear relationships among demand, weather, calendar and lag features |
| LightGBM Regressor | Machine Learning | Gradient-boosting forecasting model designed for efficient tabular learning |
| Random Forest Regressor | Tree-Based Machine Learning | Models nonlinear demand relationships using ensembles of decision trees |

Forecast outputs from these models are standardized before entering the Peak Risk classification layer.

## Peak Risk Classification Objective

The Random Forest Classifier is a separate downstream model whose objective is not to forecast electricity consumption directly.

Its objective is to estimate whether a future hourly period represents a Peak Risk event.

The architecture therefore separates two modeling problems:

1. **Demand Forecasting:** estimate future electricity consumption.
2. **Peak Risk Classification:** estimate the probability that the future period represents a Peak Risk event.

## Model-Agnostic Architecture

Seasonal Naïve ───────────┐
SARIMAX ──────────────────┤
XGBoost Regressor ────────┤
LightGBM Regressor ───────┼──> Standardized Forecast Output
Random Forest Regressor ──┘              │
                                         ↓
                              Peak Risk Feature Layer
                                         │
                       ┌─────────────────┴─────────────────┐
                       ↓                                   ↓
              Threshold Baseline                Random Forest Classifier
                       │                                   │
                       └─────────────────┬─────────────────┘
                                         ↓
                              Peak Risk Evaluation

## Objectives

1. Define a standardized interface shared by all five forecasting models.
2. Construct the Peak Risk target using thresholds derived exclusively from historical training data.
3. Establish a simple forecast-threshold Peak Risk benchmark.
4. Develop a Random Forest Classifier using information available at forecast time.
5. Address the strong class imbalance associated with rare Peak Risk events.
6. Select classifier configuration and probability threshold using validation data only.
7. Evaluate the final classifier once on the untouched 2025 test period.
8. Compare Random Forest classification against the threshold-based Peak Risk benchmark.
9. Evaluate whether classifier performance remains consistent across forecasting models and regions.

## Temporal Evaluation Principle

The 2025 test period is not used for classifier training, feature selection, probability-threshold selection, or hyperparameter tuning.

All modeling decisions are completed using training and validation data before final test evaluation.

In [4]:
# ============================================================
# 2. IMPORTS AND CONFIGURATION
# Random Forest Peak Risk Classifier
# ============================================================

# Standard library
from pathlib import Path
import gc
import time
import warnings

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.ensemble import RandomForestClassifier

# Model evaluation
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

# Reproducibility
RANDOM_STATE = 42

# Display configuration
pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.4f}"
)

# Keep warnings visible during model development
warnings.filterwarnings("default")

print("Imports loaded successfully.")
print("Random state:", RANDOM_STATE)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Imports loaded successfully.
Random state: 42
Pandas version: 2.3.3
NumPy version: 2.4.6


## 3. Standardized Forecast Interface

All forecasting models must provide their predictions using a common schema before they can be evaluated by the Peak Risk classification layer.

The standardized interface ensures that Seasonal Naïve, SARIMAX, XGBoost Regressor, LightGBM Regressor, and Random Forest Regressor can be processed through the same downstream classification pipeline.

### Required Fields

- `REGION`: study region associated with the forecast.
- `TIMESTAMP`: hourly period being predicted.
- `MODEL_NAME`: forecasting model that generated the prediction.
- `FORECAST_ORIGIN`: timestamp when the forecast would have been generated.
- `FORECAST_HORIZON_HOURS`: number of hours between forecast origin and predicted timestamp.
- `ACTUAL_CONSUMPTION`: realized electricity consumption, used only for evaluation.
- `PREDICTED_CONSUMPTION`: electricity-demand forecast generated by the upstream model.

### Design Principle

The Peak Risk Classifier consumes standardized forecast outputs rather than model-specific objects or internal model structures.

This separation allows the classifier to remain independent from the forecasting algorithm and supports consistent evaluation across all forecasting approaches.

In [1]:
# Standard forecast contract

REQUIRED_FORECAST_COLUMNS = [
    "REGION",
    "TIMESTAMP",
    "MODEL_NAME",
    "FORECAST_ORIGIN",
    "FORECAST_HORIZON_HOURS",
    "ACTUAL_CONSUMPTION",
    "PREDICTED_CONSUMPTION"
]

FORECAST_MODELS = [
    "SEASONAL_NAIVE",
    "SARIMAX",
    "XGBOOST_REGRESSOR",
    "LIGHTGBM_REGRESSOR",
    "RANDOM_FOREST_REGRESSOR"
]

REGIONS = [
    "DOWNTOWN",
    "AIRPORT_WEST"
]

print("Required forecast columns:")
for column in REQUIRED_FORECAST_COLUMNS:
    print("-", column)

print("\nSupported forecasting models:")
for model in FORECAST_MODELS:
    print("-", model)

print("\nRegions:")
for region in REGIONS:
    print("-", region)

Required forecast columns:
- REGION
- TIMESTAMP
- MODEL_NAME
- FORECAST_ORIGIN
- FORECAST_HORIZON_HOURS
- ACTUAL_CONSUMPTION
- PREDICTED_CONSUMPTION

Supported forecasting models:
- SEASONAL_NAIVE
- SARIMAX
- XGBOOST_REGRESSOR
- LIGHTGBM_REGRESSOR
- RANDOM_FOREST_REGRESSOR

Regions:
- DOWNTOWN
- AIRPORT_WEST


## 4. Forecast Contract Validation

Before forecast predictions can enter the Peak Risk pipeline, they must satisfy a common set of structural and temporal requirements.

The validation layer verifies:

- Presence of all required fields.
- Valid region and forecasting model identifiers.
- Valid timestamp formats.
- Numeric consumption predictions.
- Valid forecast horizons.
- Forecast origin occurring before the predicted timestamp.
- Absence of duplicated model-region-timestamp forecasts.
- Absence of missing values in required fields.

Forecasts that fail these checks are rejected before Peak Risk classification.

In [6]:
def validate_forecast_contract(
    df,
    required_columns=REQUIRED_FORECAST_COLUMNS,
    allowed_models=FORECAST_MODELS,
    allowed_regions=REGIONS
):
    """
    Validate standardized forecasting output before it enters
    the Peak Risk classification pipeline.
    """

    data = df.copy()

    # ---------------------------------------------------------
    # 1. Required columns
    # ---------------------------------------------------------
    missing_columns = [
        col for col in required_columns
        if col not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    # ---------------------------------------------------------
    # 2. Standardize timestamps
    # ---------------------------------------------------------
    data["TIMESTAMP"] = pd.to_datetime(
        data["TIMESTAMP"],
        errors="coerce"
    )

    data["FORECAST_ORIGIN"] = pd.to_datetime(
        data["FORECAST_ORIGIN"],
        errors="coerce"
    )

    if data["TIMESTAMP"].isna().any():
        raise ValueError(
            "Invalid values detected in TIMESTAMP."
        )

    if data["FORECAST_ORIGIN"].isna().any():
        raise ValueError(
            "Invalid values detected in FORECAST_ORIGIN."
        )

    # ---------------------------------------------------------
    # 3. Validate model identifiers
    # ---------------------------------------------------------
    invalid_models = sorted(
        set(data["MODEL_NAME"].dropna())
        - set(allowed_models)
    )

    if invalid_models:
        raise ValueError(
            f"Unsupported MODEL_NAME values: {invalid_models}"
        )

    # ---------------------------------------------------------
    # 4. Validate regions
    # ---------------------------------------------------------
    invalid_regions = sorted(
        set(data["REGION"].dropna())
        - set(allowed_regions)
    )

    if invalid_regions:
        raise ValueError(
            f"Unsupported REGION values: {invalid_regions}"
        )

    # ---------------------------------------------------------
    # 5. Required values cannot be missing
    # ---------------------------------------------------------
    missing_values = (
        data[required_columns]
        .isna()
        .sum()
    )

    missing_values = missing_values[
        missing_values > 0
    ]

    if not missing_values.empty:
        raise ValueError(
            "Missing required values:\n"
            f"{missing_values}"
        )

    # ---------------------------------------------------------
    # 6. Numeric validation
    # ---------------------------------------------------------
    numeric_columns = [
        "FORECAST_HORIZON_HOURS",
        "ACTUAL_CONSUMPTION",
        "PREDICTED_CONSUMPTION"
    ]

    for column in numeric_columns:

        data[column] = pd.to_numeric(
            data[column],
            errors="coerce"
        )

        if data[column].isna().any():
            raise ValueError(
                f"Non-numeric values detected in {column}."
            )

    # ---------------------------------------------------------
    # 7. Forecast horizon must be positive
    # ---------------------------------------------------------
    if (
        data["FORECAST_HORIZON_HOURS"] <= 0
    ).any():
        raise ValueError(
            "FORECAST_HORIZON_HOURS must be greater than zero."
        )

    # ---------------------------------------------------------
    # 8. Forecast origin must precede target timestamp
    # ---------------------------------------------------------
    invalid_temporal_order = (
        data["FORECAST_ORIGIN"]
        >= data["TIMESTAMP"]
    )

    if invalid_temporal_order.any():
        raise ValueError(
            "FORECAST_ORIGIN must occur before TIMESTAMP."
        )

    # ---------------------------------------------------------
    # 9. Check horizon consistency
    # ---------------------------------------------------------
    calculated_horizon = (
        data["TIMESTAMP"]
        - data["FORECAST_ORIGIN"]
    ).dt.total_seconds() / 3600

    horizon_mismatch = ~np.isclose(
        calculated_horizon,
        data["FORECAST_HORIZON_HOURS"]
    )

    if horizon_mismatch.any():
        raise ValueError(
            "FORECAST_HORIZON_HOURS is inconsistent "
            "with TIMESTAMP and FORECAST_ORIGIN."
        )

    # ---------------------------------------------------------
    # 10. Duplicate forecasts
    # ---------------------------------------------------------
    duplicate_mask = data.duplicated(
        subset=[
            "REGION",
            "MODEL_NAME",
            "TIMESTAMP"
        ],
        keep=False
    )

    if duplicate_mask.any():
        raise ValueError(
            "Duplicate forecasts detected for the same "
            "REGION, MODEL_NAME and TIMESTAMP."
        )

    # ---------------------------------------------------------
    # Contract passed
    # ---------------------------------------------------------
    print("Forecast contract validation: PASSED")
    print("Rows:", len(data))
    print(
        "Models:",
        sorted(data["MODEL_NAME"].unique())
    )
    print(
        "Regions:",
        sorted(data["REGION"].unique())
    )
    print(
        "Period:",
        data["TIMESTAMP"].min(),
        "to",
        data["TIMESTAMP"].max()
    )

    return data

In [7]:
# Test the forecast contract with a valid example

contract_test_valid = pd.DataFrame({
    "REGION": [
        "DOWNTOWN",
        "DOWNTOWN"
    ],
    "TIMESTAMP": [
        "2025-01-01 01:00:00",
        "2025-01-01 02:00:00"
    ],
    "MODEL_NAME": [
        "SARIMAX",
        "SARIMAX"
    ],
    "FORECAST_ORIGIN": [
        "2025-01-01 00:00:00",
        "2025-01-01 00:00:00"
    ],
    "FORECAST_HORIZON_HOURS": [
        1,
        2
    ],
    "ACTUAL_CONSUMPTION": [
        32000,
        32500
    ],
    "PREDICTED_CONSUMPTION": [
        31800,
        32700
    ]
})

contract_test_validated = (
    validate_forecast_contract(
        contract_test_valid
    )
)

Forecast contract validation: PASSED
Rows: 2
Models: ['SARIMAX']
Regions: ['DOWNTOWN']
Period: 2025-01-01 01:00:00 to 2025-01-01 02:00:00


In [8]:
# ============================================================
# 5. CONTRACT TEST — INVALID TEMPORAL ORDER
# ============================================================

contract_test_invalid = (
    contract_test_valid.copy()
)

# Deliberately invalid:
# forecast origin occurs after the target timestamp
contract_test_invalid.loc[
    0,
    "FORECAST_ORIGIN"
] = "2025-01-01 02:00:00"

try:

    validate_forecast_contract(
        contract_test_invalid
    )

    print(
        "ERROR: Invalid forecast was not rejected."
    )

except ValueError as error:

    print("Expected validation error:")
    print(error)

Expected validation error:
FORECAST_ORIGIN must occur before TIMESTAMP.


## 6. Project Paths and Forecast Adapter Layer

Forecasting models may produce prediction files with different internal column names or output structures.

The adapter layer converts model-specific forecast outputs into the standardized forecast contract before Peak Risk processing.

This design keeps model-specific transformations outside the Random Forest classification logic.

The classification pipeline therefore receives the same standardized structure regardless of whether predictions originate from:

- Seasonal Naïve
- SARIMAX
- XGBoost Regressor
- LightGBM Regressor
- Random Forest Regressor

Model-specific adapters are responsible only for translating forecasting outputs into the common interface. They do not modify forecast values or classification targets.

In [9]:
# ============================================================
# 6. PROJECT PATHS
# ============================================================

DATA_PATH = Path("../data")
PROCESSED_PATH = DATA_PATH / "processed"

MODEL_OUTPUT_PATH = (
    PROCESSED_PATH / "model_outputs"
)

PEAK_RISK_OUTPUT_PATH = (
    PROCESSED_PATH / "peak_risk_outputs"
)

PEAK_RISK_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Processed data path:")
print(PROCESSED_PATH.resolve())

print("\nForecast model outputs:")
print(MODEL_OUTPUT_PATH.resolve())

print("\nPeak Risk outputs:")
print(PEAK_RISK_OUTPUT_PATH.resolve())

Processed data path:
C:\Users\ASUS\Documents\Master of Data Analytics\Capstone\ontario-electricity-peak-risk\data\processed

Forecast model outputs:
C:\Users\ASUS\Documents\Master of Data Analytics\Capstone\ontario-electricity-peak-risk\data\processed\model_outputs

Peak Risk outputs:
C:\Users\ASUS\Documents\Master of Data Analytics\Capstone\ontario-electricity-peak-risk\data\processed\peak_risk_outputs


In [10]:
# ============================================================
# AVAILABLE FORECAST OUTPUTS
# ============================================================

if MODEL_OUTPUT_PATH.exists():

    available_files = sorted(
        MODEL_OUTPUT_PATH.glob("*.csv")
    )

    print(
        "Forecast output files found:",
        len(available_files)
    )

    for file in available_files:
        print("-", file.name)

else:

    print(
        "Forecast model output directory does not exist yet."
    )

Forecast output files found: 19
- airport_west_c1_calendar_validation_metrics.csv
- airport_west_c1_calendar_validation_predictions.csv
- airport_west_c2_calendar_validation_metrics.csv
- airport_west_c2_calendar_validation_predictions.csv
- airport_west_c3_calendar_validation_predictions.csv
- airport_west_c3_calendar_weather_validation_metrics.csv
- airport_west_c3_calendar_weather_validation_predictions.csv
- airport_west_final_c3_calendar_weather_2025_test_metrics.csv
- airport_west_final_c3_calendar_weather_2025_test_predictions.csv
- downtown_c1_calendar_validation_metrics.csv
- downtown_c1_calendar_validation_predictions.csv
- downtown_c2_calendar_validation_metrics.csv
- downtown_c2_calendar_validation_predictions.csv
- downtown_c3_calendar_validation_metrics.csv
- downtown_c3_calendar_validation_predictions.csv
- downtown_c3_calendar_weather_validation_metrics.csv
- downtown_c3_calendar_weather_validation_predictions.csv
- downtown_final_c3_calendar_weather_2025_test_metrics.c

In [11]:
# ============================================================
# 7. INSPECT SARIMAX VALIDATION FORECAST OUTPUTS
# ============================================================

SARIMAX_VALIDATION_FILES = {
    "DOWNTOWN": (
        MODEL_OUTPUT_PATH /
        "downtown_c3_calendar_weather_validation_predictions.csv"
    ),
    "AIRPORT_WEST": (
        MODEL_OUTPUT_PATH /
        "airport_west_c3_calendar_weather_validation_predictions.csv"
    )
}

sarimax_raw_validation = {}

for region, file_path in SARIMAX_VALIDATION_FILES.items():

    df = pd.read_csv(
        file_path,
        parse_dates=["TIMESTAMP"]
    )

    sarimax_raw_validation[region] = df

    print("=" * 60)
    print(region)
    print("File:", file_path.name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(
        "Period:",
        df["TIMESTAMP"].min(),
        "to",
        df["TIMESTAMP"].max()
    )

    display(df.head(3))

DOWNTOWN
File: downtown_c3_calendar_weather_validation_predictions.csv
Shape: (4416, 3)
Columns: ['TIMESTAMP', 'ACTUAL', 'PREDICTED']
Period: 2024-07-01 00:00:00 to 2024-12-31 23:00:00


,TIMESTAMP,ACTUAL,PREDICTED
0,2024-07-01 00:00:00,"15,044.5000","14,978.9068"
1,2024-07-01 01:00:00,"13,848.9000","13,810.5505"
2,2024-07-01 02:00:00,"12,954.8000","13,016.7981"


AIRPORT_WEST
File: airport_west_c3_calendar_weather_validation_predictions.csv
Shape: (4416, 3)
Columns: ['TIMESTAMP', 'ACTUAL', 'PREDICTED']
Period: 2024-07-01 00:00:00 to 2024-12-31 23:00:00


,TIMESTAMP,ACTUAL,PREDICTED
0,2024-07-01 00:00:00,"21,864.8000","21,226.1544"
1,2024-07-01 01:00:00,"20,008.0000","19,081.4360"
2,2024-07-01 02:00:00,"18,654.9000","17,816.0799"


## 8. SARIMAX Forecast Adapter

The SARIMAX validation outputs contain three native fields: `TIMESTAMP`, `ACTUAL`, and `PREDICTED`.

Because validation was performed using sequential 24-hour rolling forecast blocks, forecast metadata can be reconstructed deterministically.

For each 24-hour block:

- The forecast origin is the hour immediately preceding the first predicted hour.
- Forecast horizons range from 1 through 24 hours.
- After each block, actual observations become available before the next 24-hour forecast is generated.

The adapter converts the native SARIMAX output into the standardized forecast contract without modifying actual or predicted consumption values.

In [12]:
# ============================================================
# 8. SARIMAX FORECAST ADAPTER
# ============================================================

def adapt_sarimax_rolling_forecast(
    df,
    region,
    horizon=24
):
    """
    Convert SARIMAX rolling forecast output into the
    standardized forecast contract.

    Assumes sequential rolling forecast blocks of fixed
    length, with horizons 1 through `horizon`.
    """

    data = (
        df.copy()
        .sort_values("TIMESTAMP")
        .reset_index(drop=True)
    )

    # Validate native SARIMAX structure
    required_native_columns = [
        "TIMESTAMP",
        "ACTUAL",
        "PREDICTED"
    ]

    missing = [
        col
        for col in required_native_columns
        if col not in data.columns
    ]

    if missing:
        raise ValueError(
            f"SARIMAX output missing columns: {missing}"
        )

    # Ensure timestamp type
    data["TIMESTAMP"] = pd.to_datetime(
        data["TIMESTAMP"]
    )

    # Standard identifiers
    data["REGION"] = region
    data["MODEL_NAME"] = "SARIMAX"

    # Reconstruct horizons:
    # 1, 2, ..., 24 for each rolling block
    data["FORECAST_HORIZON_HOURS"] = (
        np.arange(len(data)) % horizon
    ) + 1

    # Forecast origin is determined from:
    # target timestamp - forecast horizon
    data["FORECAST_ORIGIN"] = (
        data["TIMESTAMP"]
        - pd.to_timedelta(
            data["FORECAST_HORIZON_HOURS"],
            unit="h"
        )
    )

    # Standardize consumption column names
    data = data.rename(
        columns={
            "ACTUAL": "ACTUAL_CONSUMPTION",
            "PREDICTED": "PREDICTED_CONSUMPTION"
        }
    )

    # Keep only standardized contract fields
    data = data[
        REQUIRED_FORECAST_COLUMNS
    ].copy()

    # Validate resulting contract
    data = validate_forecast_contract(
        data
    )

    return data

In [13]:
# Adapt SARIMAX validation forecasts
# for both study regions

sarimax_validation_standardized = []

for region in REGIONS:

    standardized = (
        adapt_sarimax_rolling_forecast(
            df=sarimax_raw_validation[region],
            region=region,
            horizon=24
        )
    )

    sarimax_validation_standardized.append(
        standardized
    )

sarimax_validation_standardized = pd.concat(
    sarimax_validation_standardized,
    ignore_index=True
)

print("=" * 60)
print("COMBINED SARIMAX VALIDATION FORECASTS")
print("=" * 60)

print(
    "Shape:",
    sarimax_validation_standardized.shape
)

print(
    "Rows by region:"
)

print(
    sarimax_validation_standardized[
        "REGION"
    ].value_counts()
)

display(
    sarimax_validation_standardized.head(26)
)

Forecast contract validation: PASSED
Rows: 4416
Models: ['SARIMAX']
Regions: ['DOWNTOWN']
Period: 2024-07-01 00:00:00 to 2024-12-31 23:00:00
Forecast contract validation: PASSED
Rows: 4416
Models: ['SARIMAX']
Regions: ['AIRPORT_WEST']
Period: 2024-07-01 00:00:00 to 2024-12-31 23:00:00
COMBINED SARIMAX VALIDATION FORECASTS
Shape: (8832, 7)
Rows by region:
REGION
DOWNTOWN        4416
AIRPORT_WEST    4416
Name: count, dtype: int64


,REGION,TIMESTAMP,MODEL_NAME,FORECAST_ORIGIN,FORECAST_HORIZON_HOURS,ACTUAL_CONSUMPTION,PREDICTED_CONSUMPTION
0,DOWNTOWN,2024-07-01 00:00:00,SARIMAX,2024-06-30 23:00:00,1,"15,044.5000","14,978.9068"
1,DOWNTOWN,2024-07-01 01:00:00,SARIMAX,2024-06-30 23:00:00,2,"13,848.9000","13,810.5505"
2,DOWNTOWN,2024-07-01 02:00:00,SARIMAX,2024-06-30 23:00:00,3,"12,954.8000","13,016.7981"
3,DOWNTOWN,2024-07-01 03:00:00,SARIMAX,2024-06-30 23:00:00,4,"12,501.9000","12,590.0243"
4,DOWNTOWN,2024-07-01 04:00:00,SARIMAX,2024-06-30 23:00:00,5,"12,287.7000","12,531.6978"
5,DOWNTOWN,2024-07-01 05:00:00,SARIMAX,2024-06-30 23:00:00,6,"12,650.2000","12,911.9374"
6,DOWNTOWN,2024-07-01 06:00:00,SARIMAX,2024-06-30 23:00:00,7,"13,641.6000","14,180.8951"
7,DOWNTOWN,2024-07-01 07:00:00,SARIMAX,2024-06-30 23:00:00,8,"15,397.0000","16,352.8031"
8,DOWNTOWN,2024-07-01 08:00:00,SARIMAX,2024-06-30 23:00:00,9,"17,228.5000","17,843.6318"
9,DOWNTOWN,2024-07-01 09:00:00,SARIMAX,2024-06-30 23:00:00,10,"19,055.0000","18,927.6080"


In [14]:
# ============================================================
# 8.2 VERIFY ROLLING FORECAST BLOCK TRANSITION
# ============================================================

display(
    sarimax_validation_standardized[
        sarimax_validation_standardized["REGION"] == "DOWNTOWN"
    ].iloc[22:26]
)

,REGION,TIMESTAMP,MODEL_NAME,FORECAST_ORIGIN,FORECAST_HORIZON_HOURS,ACTUAL_CONSUMPTION,PREDICTED_CONSUMPTION
22,DOWNTOWN,2024-07-01 22:00:00,SARIMAX,2024-06-30 23:00:00,23,"20,838.2000","17,828.0353"
23,DOWNTOWN,2024-07-01 23:00:00,SARIMAX,2024-06-30 23:00:00,24,"18,402.9000","15,973.2828"
24,DOWNTOWN,2024-07-02 00:00:00,SARIMAX,2024-07-01 23:00:00,1,"16,191.6000","16,731.1234"
25,DOWNTOWN,2024-07-02 01:00:00,SARIMAX,2024-07-01 23:00:00,2,"14,863.4000","15,319.0513"


## 9. Peak Risk Target Definition

Peak Risk is defined using a region-specific electricity-demand threshold derived exclusively from the historical training period.

For each region, an hourly observation is labeled as a Peak Risk event when actual electricity consumption is greater than or equal to the region-specific 97.5th percentile threshold.

The target is therefore defined as:

`ACTUAL_PEAK_RISK = 1` if `ACTUAL_CONSUMPTION >= PEAK_THRESHOLD`

and:

`ACTUAL_PEAK_RISK = 0` otherwise.

Thresholds are calculated only from training data to prevent information from validation or test periods from influencing the target definition.

Because the 97.5th percentile identifies rare high-demand conditions, Peak Risk is expected to be strongly imbalanced, with substantially fewer positive observations than negative observations.

In [15]:
# ============================================================
# 9. PEAK RISK THRESHOLDS
# Derived from training data only
# ============================================================

PEAK_RISK_THRESHOLDS = {
    "DOWNTOWN": 34974.11,
    "AIRPORT_WEST": 48865.85
}

for region, threshold in PEAK_RISK_THRESHOLDS.items():

    print(
        region,
        "| P97.5 threshold:",
        f"{threshold:,.2f}"
    )

DOWNTOWN | P97.5 threshold: 34,974.11
AIRPORT_WEST | P97.5 threshold: 48,865.85


In [16]:
# ============================================================
# 9.1 ADD ACTUAL PEAK RISK TARGET
# ============================================================

def add_actual_peak_risk_target(
    df,
    thresholds=PEAK_RISK_THRESHOLDS
):
    """
    Add region-specific actual Peak Risk target.
    """

    data = df.copy()

    data["PEAK_THRESHOLD"] = (
        data["REGION"].map(thresholds)
    )

    if data["PEAK_THRESHOLD"].isna().any():
        raise ValueError(
            "Missing Peak Risk threshold for one or more regions."
        )

    data["ACTUAL_PEAK_RISK"] = (
        data["ACTUAL_CONSUMPTION"]
        >= data["PEAK_THRESHOLD"]
    ).astype(int)

    return data

In [17]:
sarimax_validation_peak = (
    add_actual_peak_risk_target(
        sarimax_validation_standardized
    )
)

display(
    sarimax_validation_peak.head()
)

,REGION,TIMESTAMP,MODEL_NAME,FORECAST_ORIGIN,FORECAST_HORIZON_HOURS,ACTUAL_CONSUMPTION,PREDICTED_CONSUMPTION,PEAK_THRESHOLD,ACTUAL_PEAK_RISK
0,DOWNTOWN,2024-07-01 00:00:00,SARIMAX,2024-06-30 23:00:00,1,"15,044.5000","14,978.9068","34,974.1100",0
1,DOWNTOWN,2024-07-01 01:00:00,SARIMAX,2024-06-30 23:00:00,2,"13,848.9000","13,810.5505","34,974.1100",0
2,DOWNTOWN,2024-07-01 02:00:00,SARIMAX,2024-06-30 23:00:00,3,"12,954.8000","13,016.7981","34,974.1100",0
3,DOWNTOWN,2024-07-01 03:00:00,SARIMAX,2024-06-30 23:00:00,4,"12,501.9000","12,590.0243","34,974.1100",0
4,DOWNTOWN,2024-07-01 04:00:00,SARIMAX,2024-06-30 23:00:00,5,"12,287.7000","12,531.6978","34,974.1100",0


In [18]:
# ============================================================
# 9.2 PEAK RISK CLASS DISTRIBUTION
# ============================================================

peak_risk_distribution = (
    sarimax_validation_peak
    .groupby("REGION")["ACTUAL_PEAK_RISK"]
    .agg(
        TOTAL_HOURS="count",
        PEAK_HOURS="sum"
    )
    .reset_index()
)

peak_risk_distribution["NON_PEAK_HOURS"] = (
    peak_risk_distribution["TOTAL_HOURS"]
    - peak_risk_distribution["PEAK_HOURS"]
)

peak_risk_distribution["PEAK_RATE_PERCENT"] = (
    peak_risk_distribution["PEAK_HOURS"]
    / peak_risk_distribution["TOTAL_HOURS"]
    * 100
)

peak_risk_distribution[
    "NON_PEAK_RATE_PERCENT"
] = (
    peak_risk_distribution["NON_PEAK_HOURS"]
    / peak_risk_distribution["TOTAL_HOURS"]
    * 100
)

peak_risk_distribution.round(2)

,REGION,TOTAL_HOURS,PEAK_HOURS,NON_PEAK_HOURS,PEAK_RATE_PERCENT,NON_PEAK_RATE_PERCENT
0,AIRPORT_WEST,4416,378,4038,8.5600,91.4400
1,DOWNTOWN,4416,166,4250,3.7600,96.2400


In [19]:
# ============================================================
# 9.3 INSPECT ACTUAL PEAK RISK EVENTS
# ============================================================

for region in REGIONS:

    print("=" * 60)
    print(region)

    peak_events = (
        sarimax_validation_peak[
            (sarimax_validation_peak["REGION"] == region) &
            (sarimax_validation_peak["ACTUAL_PEAK_RISK"] == 1)
        ]
        .sort_values(
            "ACTUAL_CONSUMPTION",
            ascending=False
        )
    )

    print(
        "Peak Risk observations:",
        len(peak_events)
    )

    display(
        peak_events[
            [
                "TIMESTAMP",
                "ACTUAL_CONSUMPTION",
                "PEAK_THRESHOLD",
                "PREDICTED_CONSUMPTION",
                "FORECAST_HORIZON_HOURS"
            ]
        ].head(5)
    )

DOWNTOWN
Peak Risk observations: 166


,TIMESTAMP,ACTUAL_CONSUMPTION,PEAK_THRESHOLD,PREDICTED_CONSUMPTION,FORECAST_HORIZON_HOURS
736,2024-07-31 16:00:00,"43,439.2000","34,974.1100","42,012.5944",17
738,2024-07-31 18:00:00,"42,982.2000","34,974.1100","41,903.9181",19
1386,2024-08-27 18:00:00,"42,845.9000","34,974.1100","37,551.8237",19
1385,2024-08-27 17:00:00,"42,772.4000","34,974.1100","38,193.3026",18
737,2024-07-31 17:00:00,"42,749.9000","34,974.1100","42,211.6597",18


AIRPORT_WEST
Peak Risk observations: 378


,TIMESTAMP,ACTUAL_CONSUMPTION,PEAK_THRESHOLD,PREDICTED_CONSUMPTION,FORECAST_HORIZON_HOURS
5151,2024-07-31 15:00:00,"65,999.1000","48,865.8500","62,766.8702",16
5152,2024-07-31 16:00:00,"65,937.6000","48,865.8500","61,533.3096",17
5176,2024-08-01 16:00:00,"65,189.9000","48,865.8500","65,457.2455",17
5104,2024-07-29 16:00:00,"65,037.1000","48,865.8500","63,517.5797",17
5175,2024-08-01 15:00:00,"64,864.3000","48,865.8500","65,883.6363",16


## 10. Threshold-Based Peak Risk Baseline

Before developing the Random Forest Classifier, a simple Peak Risk classification baseline is established directly from forecasted electricity consumption.

For each region:

`PREDICTED_PEAK_RISK = 1`

when:

`PREDICTED_CONSUMPTION >= PEAK_THRESHOLD`

Otherwise:

`PREDICTED_PEAK_RISK = 0`

This approach represents the simplest way to transform an electricity-demand forecast into a Peak Risk warning.

The Random Forest Classifier must demonstrate additional classification value relative to this baseline. Because Peak Risk events are relatively rare, evaluation focuses primarily on Precision, Recall, F1-score, and the number of missed Peak Risk events rather than overall accuracy.

In [20]:
# ============================================================
# 10.1 THRESHOLD-BASED PEAK RISK BASELINE
# ============================================================

sarimax_validation_peak[
    "BASELINE_PEAK_RISK"
] = (
    sarimax_validation_peak[
        "PREDICTED_CONSUMPTION"
    ]
    >=
    sarimax_validation_peak[
        "PEAK_THRESHOLD"
    ]
).astype(int)

print(
    sarimax_validation_peak[
        [
            "ACTUAL_PEAK_RISK",
            "BASELINE_PEAK_RISK"
        ]
    ].value_counts()
)

ACTUAL_PEAK_RISK  BASELINE_PEAK_RISK
0                 0                     8141
1                 1                      392
                  0                      152
0                 1                      147
Name: count, dtype: int64


In [21]:
# ============================================================
# 10.2 BASELINE CLASSIFICATION PERFORMANCE BY REGION
# ============================================================

baseline_results = []

for region in REGIONS:

    region_data = sarimax_validation_peak[
        sarimax_validation_peak["REGION"] == region
    ].copy()

    y_true = region_data[
        "ACTUAL_PEAK_RISK"
    ]

    y_pred = region_data[
        "BASELINE_PEAK_RISK"
    ]

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    baseline_results.append({
        "REGION": region,
        "MODEL_NAME": "SARIMAX",
        "METHOD": "P97.5 Threshold Baseline",
        "TRUE_NEGATIVES": tn,
        "FALSE_POSITIVES": fp,
        "FALSE_NEGATIVES": fn,
        "TRUE_POSITIVES": tp,
        "PRECISION": precision,
        "RECALL": recall,
        "F1_SCORE": f1
    })

baseline_results = pd.DataFrame(
    baseline_results
)

baseline_results.round(4)

,REGION,MODEL_NAME,METHOD,TRUE_NEGATIVES,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES,PRECISION,RECALL,F1_SCORE
0,DOWNTOWN,SARIMAX,P97.5 Threshold Baseline,4198,52,53,113,0.6848,0.6807,0.6828
1,AIRPORT_WEST,SARIMAX,P97.5 Threshold Baseline,3943,95,99,279,0.7460,0.7381,0.7420


## 11. Random Forest Feature Engineering

The Random Forest Classifier uses only information that would be available when a forecast is generated.

Actual future electricity consumption is excluded from classifier features because it is unknown at forecast time and is used only to construct the ground-truth Peak Risk target.

The initial model-agnostic feature set includes:

### Forecast Features

- `PREDICTED_CONSUMPTION`: demand forecast produced by the upstream forecasting model.
- `FORECAST_HORIZON_HOURS`: number of hours ahead being predicted.

### Threshold-Distance Features

- `PEAK_THRESHOLD`: region-specific Peak Risk threshold derived from training data.
- `DISTANCE_TO_THRESHOLD`: predicted consumption minus the Peak Risk threshold.
- `PREDICTED_LOAD_RATIO`: predicted consumption divided by the Peak Risk threshold.

### Calendar Features

- `HOUR`
- `DAY_OF_WEEK`
- `MONTH`
- `IS_WEEKEND`

This initial feature set is intentionally model-agnostic so that the same classification architecture can consume forecasts generated by Seasonal Naïve, SARIMAX, XGBoost Regressor, LightGBM Regressor, and Random Forest Regressor.

`ACTUAL_CONSUMPTION` is retained exclusively for target construction and evaluation and is never used as a classifier input.

In [22]:
# ============================================================
# 11.1 RANDOM FOREST FEATURE ENGINEERING
# ============================================================

def build_peak_risk_features(df):

    data = df.copy()

    # --------------------------------------------------------
    # Forecast-to-threshold features
    # --------------------------------------------------------

    data["DISTANCE_TO_THRESHOLD"] = (
        data["PREDICTED_CONSUMPTION"]
        - data["PEAK_THRESHOLD"]
    )

    data["PREDICTED_LOAD_RATIO"] = (
        data["PREDICTED_CONSUMPTION"]
        / data["PEAK_THRESHOLD"]
    )

    # --------------------------------------------------------
    # Calendar features available at forecast time
    # --------------------------------------------------------

    data["HOUR"] = (
        data["TIMESTAMP"].dt.hour
    )

    data["DAY_OF_WEEK"] = (
        data["TIMESTAMP"].dt.dayofweek
    )

    data["MONTH"] = (
        data["TIMESTAMP"].dt.month
    )

    data["IS_WEEKEND"] = (
        data["DAY_OF_WEEK"] >= 5
    ).astype(int)

    return data

In [23]:
sarimax_validation_features = (
    build_peak_risk_features(
        sarimax_validation_peak
    )
)

RF_FEATURES = [
    "PREDICTED_CONSUMPTION",
    "PEAK_THRESHOLD",
    "DISTANCE_TO_THRESHOLD",
    "PREDICTED_LOAD_RATIO",
    "FORECAST_HORIZON_HOURS",
    "HOUR",
    "DAY_OF_WEEK",
    "MONTH",
    "IS_WEEKEND"
]

print("Random Forest features:")
for feature in RF_FEATURES:
    print("-", feature)

print(
    "\nMissing feature values:",
    sarimax_validation_features[
        RF_FEATURES
    ].isna().sum().sum()
)

display(
    sarimax_validation_features[
        [
            "REGION",
            "TIMESTAMP",
            *RF_FEATURES,
            "ACTUAL_PEAK_RISK"
        ]
    ].head()
)

Random Forest features:
- PREDICTED_CONSUMPTION
- PEAK_THRESHOLD
- DISTANCE_TO_THRESHOLD
- PREDICTED_LOAD_RATIO
- FORECAST_HORIZON_HOURS
- HOUR
- DAY_OF_WEEK
- MONTH
- IS_WEEKEND

Missing feature values: 0


,REGION,TIMESTAMP,PREDICTED_CONSUMPTION,PEAK_THRESHOLD,DISTANCE_TO_THRESHOLD,PREDICTED_LOAD_RATIO,FORECAST_HORIZON_HOURS,HOUR,DAY_OF_WEEK,MONTH,IS_WEEKEND,ACTUAL_PEAK_RISK
0,DOWNTOWN,2024-07-01 00:00:00,"14,978.9068","34,974.1100","-19,995.2032",0.4283,1,0,0,7,0,0
1,DOWNTOWN,2024-07-01 01:00:00,"13,810.5505","34,974.1100","-21,163.5595",0.3949,2,1,0,7,0,0
2,DOWNTOWN,2024-07-01 02:00:00,"13,016.7981","34,974.1100","-21,957.3119",0.3722,3,2,0,7,0,0
3,DOWNTOWN,2024-07-01 03:00:00,"12,590.0243","34,974.1100","-22,384.0857",0.3600,4,3,0,7,0,0
4,DOWNTOWN,2024-07-01 04:00:00,"12,531.6978","34,974.1100","-22,442.4122",0.3583,5,4,0,7,0,0


## 12. Model-Specific Random Forest Strategy

The Peak Risk module uses the same Random Forest classification architecture independently for each upstream forecasting model.

A separate classifier is trained and evaluated for:

- Seasonal Naïve
- SARIMAX
- XGBoost Regressor
- LightGBM Regressor
- Random Forest Regressor

The classifiers share:

- the same standardized forecast contract;
- the same Peak Risk definition;
- the same feature engineering procedure;
- the same temporal train-validation-test design;
- the same class-imbalance strategy;
- the same hyperparameter-selection procedure;
- the same probability-threshold selection procedure;
- the same evaluation metrics.

However, each classifier is trained only with forecasts generated by its corresponding forecasting model.

This prevents forecast-error patterns from different models from being mixed during classifier training and allows a fair comparison of how effectively each forecasting approach supports downstream Peak Risk detection.

In [24]:
# ============================================================
# 13.1 CLASSIFIER TEMPORAL SPLIT
# ============================================================

CLASSIFIER_TRAIN_START = pd.Timestamp(
    "2024-01-01 00:00:00"
)

CLASSIFIER_TRAIN_END = pd.Timestamp(
    "2024-06-30 23:00:00"
)

CLASSIFIER_VALIDATION_START = pd.Timestamp(
    "2024-07-01 00:00:00"
)

CLASSIFIER_VALIDATION_END = pd.Timestamp(
    "2024-12-31 23:00:00"
)

CLASSIFIER_TEST_START = pd.Timestamp(
    "2025-01-01 00:00:00"
)

CLASSIFIER_TEST_END = pd.Timestamp(
    "2025-12-31 23:00:00"
)

print(
    "Classifier Train:",
    CLASSIFIER_TRAIN_START,
    "to",
    CLASSIFIER_TRAIN_END
)

print(
    "Classifier Validation:",
    CLASSIFIER_VALIDATION_START,
    "to",
    CLASSIFIER_VALIDATION_END
)

print(
    "Classifier Test:",
    CLASSIFIER_TEST_START,
    "to",
    CLASSIFIER_TEST_END
)

Classifier Train: 2024-01-01 00:00:00 to 2024-06-30 23:00:00
Classifier Validation: 2024-07-01 00:00:00 to 2024-12-31 23:00:00
Classifier Test: 2025-01-01 00:00:00 to 2025-12-31 23:00:00


## 14. Load and Standardize SARIMAX Classifier Training Forecasts

The Random Forest classifier training set is built from historical out-of-sample demand forecasts.

For SARIMAX, January–June 2024 forecasts were generated using models trained only on data available through December 31, 2023.

This preserves temporal integrity and prevents classifier training from using in-sample or future-informed forecast values.

The same SARIMAX adapter used for validation is applied to classifier-training forecasts so that training and validation share the same standardized forecast contract.

In [25]:
# ============================================================
# 14.1 LOAD SARIMAX CLASSIFIER-TRAINING FORECASTS
# ============================================================

SARIMAX_CLASSIFIER_TRAIN_FILES = {
    "DOWNTOWN": (
        MODEL_OUTPUT_PATH /
        "downtown_c3_calendar_weather_classifier_train_predictions.csv"
    ),
    "AIRPORT_WEST": (
        MODEL_OUTPUT_PATH /
        "airport_west_c3_calendar_weather_classifier_train_predictions.csv"
    )
}

sarimax_raw_classifier_train = {}

for region, file_path in SARIMAX_CLASSIFIER_TRAIN_FILES.items():

    df = pd.read_csv(
        file_path,
        parse_dates=["TIMESTAMP"]
    )

    sarimax_raw_classifier_train[region] = df

    print("=" * 60)
    print(region)
    print("File:", file_path.name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    print(
        "Period:",
        df["TIMESTAMP"].min(),
        "to",
        df["TIMESTAMP"].max()
    )

DOWNTOWN
File: downtown_c3_calendar_weather_classifier_train_predictions.csv
Shape: (4368, 3)
Columns: ['TIMESTAMP', 'ACTUAL', 'PREDICTED']
Period: 2024-01-01 00:00:00 to 2024-06-30 23:00:00
AIRPORT_WEST
File: airport_west_c3_calendar_weather_classifier_train_predictions.csv
Shape: (4368, 3)
Columns: ['TIMESTAMP', 'ACTUAL', 'PREDICTED']
Period: 2024-01-01 00:00:00 to 2024-06-30 23:00:00


In [26]:
# ============================================================
# 14.2 STANDARDIZE SARIMAX CLASSIFIER-TRAINING FORECASTS
# ============================================================

sarimax_classifier_train_standardized = []

for region in REGIONS:

    standardized = adapt_sarimax_rolling_forecast(
        df=sarimax_raw_classifier_train[region],
        region=region,
        horizon=24
    )

    sarimax_classifier_train_standardized.append(
        standardized
    )

sarimax_classifier_train_standardized = pd.concat(
    sarimax_classifier_train_standardized,
    ignore_index=True
)

print("=" * 60)
print("COMBINED SARIMAX CLASSIFIER TRAINING FORECASTS")
print("=" * 60)

print(
    "Shape:",
    sarimax_classifier_train_standardized.shape
)

print(
    "Rows by region:"
)

print(
    sarimax_classifier_train_standardized[
        "REGION"
    ].value_counts()
)

print(
    "Period:",
    sarimax_classifier_train_standardized["TIMESTAMP"].min(),
    "to",
    sarimax_classifier_train_standardized["TIMESTAMP"].max()
)

Forecast contract validation: PASSED
Rows: 4368
Models: ['SARIMAX']
Regions: ['DOWNTOWN']
Period: 2024-01-01 00:00:00 to 2024-06-30 23:00:00
Forecast contract validation: PASSED
Rows: 4368
Models: ['SARIMAX']
Regions: ['AIRPORT_WEST']
Period: 2024-01-01 00:00:00 to 2024-06-30 23:00:00
COMBINED SARIMAX CLASSIFIER TRAINING FORECASTS
Shape: (8736, 7)
Rows by region:
REGION
DOWNTOWN        4368
AIRPORT_WEST    4368
Name: count, dtype: int64
Period: 2024-01-01 00:00:00 to 2024-06-30 23:00:00


In [27]:
# ============================================================
# 14.3 BUILD CLASSIFIER TRAINING TARGET AND FEATURES
# ============================================================

sarimax_classifier_train_peak = (
    add_actual_peak_risk_target(
        sarimax_classifier_train_standardized
    )
)

sarimax_classifier_train_features = (
    build_peak_risk_features(
        sarimax_classifier_train_peak
    )
)

print(
    "Training rows:",
    len(sarimax_classifier_train_features)
)

print(
    "Missing feature values:",
    sarimax_classifier_train_features[
        RF_FEATURES
    ].isna().sum().sum()
)

print(
    "\nPeak Risk distribution:"
)

print(
    sarimax_classifier_train_features
    .groupby("REGION")["ACTUAL_PEAK_RISK"]
    .agg(
        TOTAL_HOURS="count",
        PEAK_HOURS="sum"
    )
)

Training rows: 8736
Missing feature values: 0

Peak Risk distribution:
              TOTAL_HOURS  PEAK_HOURS
REGION                               
AIRPORT_WEST         4368          88
DOWNTOWN             4368          87


## 15. Random Forest Training Strategy

A separate Random Forest Peak Risk classifier is trained for each combination of forecasting model and region.

For the current SARIMAX implementation:

- SARIMAX → Downtown → Random Forest Classifier
- SARIMAX → Airport-West → Random Forest Classifier

Both classifiers use the same feature set, Random Forest architecture, temporal evaluation framework, and model-selection procedure.

Training uses January–June 2024 out-of-sample forecasts.

Validation uses July–December 2024 rolling forecasts and remains temporally separated from classifier training.

Because Peak Risk represents approximately 2% of classifier-training observations, class imbalance is explicitly addressed during model development.

Model selection emphasizes:

- Recall
- Precision
- F1-score
- PR-AUC
- False Negatives
- False Positives

Overall accuracy is not used as the primary selection criterion because the majority non-Peak class dominates the dataset.

In [28]:
# ============================================================
# 15.1 PREPARE RANDOM FOREST TRAIN / VALIDATION SETS
# ============================================================

# Validation features were already created in Section 11
rf_train_data = (
    sarimax_classifier_train_features.copy()
)

rf_validation_data = (
    sarimax_validation_features.copy()
)

rf_datasets = {}

for region in REGIONS:

    train_region = (
        rf_train_data[
            rf_train_data["REGION"] == region
        ]
        .sort_values("TIMESTAMP")
        .copy()
    )

    validation_region = (
        rf_validation_data[
            rf_validation_data["REGION"] == region
        ]
        .sort_values("TIMESTAMP")
        .copy()
    )

    X_train = train_region[
        RF_FEATURES
    ].copy()

    y_train = train_region[
        "ACTUAL_PEAK_RISK"
    ].copy()

    X_validation = validation_region[
        RF_FEATURES
    ].copy()

    y_validation = validation_region[
        "ACTUAL_PEAK_RISK"
    ].copy()

    rf_datasets[region] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_validation": X_validation,
        "y_validation": y_validation,
        "train_timestamps": train_region["TIMESTAMP"],
        "validation_timestamps": validation_region["TIMESTAMP"]
    }

    print("=" * 60)
    print(region)

    print(
        "X_train:",
        X_train.shape
    )

    print(
        "y_train:",
        y_train.shape
    )

    print(
        "X_validation:",
        X_validation.shape
    )

    print(
        "y_validation:",
        y_validation.shape
    )

    print(
        "Train Peak Risk:",
        int(y_train.sum()),
        "/",
        len(y_train),
        f"({y_train.mean() * 100:.2f}%)"
    )

    print(
        "Validation Peak Risk:",
        int(y_validation.sum()),
        "/",
        len(y_validation),
        f"({y_validation.mean() * 100:.2f}%)"
    )

    print(
        "Temporal overlap:",
        (
            rf_datasets[region]["train_timestamps"].max()
            >=
            rf_datasets[region]["validation_timestamps"].min()
        )
    )

DOWNTOWN
X_train: (4368, 9)
y_train: (4368,)
X_validation: (4416, 9)
y_validation: (4416,)
Train Peak Risk: 87 / 4368 (1.99%)
Validation Peak Risk: 166 / 4416 (3.76%)
Temporal overlap: False
AIRPORT_WEST
X_train: (4368, 9)
y_train: (4368,)
X_validation: (4416, 9)
y_validation: (4416,)
Train Peak Risk: 88 / 4368 (2.01%)
Validation Peak Risk: 378 / 4416 (8.56%)
Temporal overlap: False


## 16. Random Forest v1 — Baseline Classifier

An initial Random Forest classifier is trained independently for each region using the standardized SARIMAX forecast features.

This first model establishes a machine-learning classification baseline before hyperparameter or probability-threshold optimization.

Because Peak Risk is strongly imbalanced in the training period, `class_weight="balanced"` is used to increase the relative importance of the minority Peak Risk class.

The initial probability threshold remains fixed at 0.50.

No validation-driven tuning is performed at this stage.

In [29]:
# ============================================================
# 16.1 RANDOM FOREST V1 — TRAIN AND VALIDATE
# ============================================================

rf_v1_models = {}
rf_v1_predictions = {}
rf_v1_results = []

for region in REGIONS:

    data = rf_datasets[region]

    X_train = data["X_train"]
    y_train = data["y_train"]

    X_validation = data["X_validation"]
    y_validation = data["y_validation"]

    # --------------------------------------------------------
    # Baseline Random Forest
    # --------------------------------------------------------

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    fit_time_seconds = (
        time.time() - start_time
    )

    # --------------------------------------------------------
    # Validation probabilities and predictions
    # --------------------------------------------------------

    y_probability = model.predict_proba(
        X_validation
    )[:, 1]

    y_pred = (
        y_probability >= 0.50
    ).astype(int)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        y_pred,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_validation,
        y_probability
    )

    pr_auc = average_precision_score(
        y_validation,
        y_probability
    )

    # --------------------------------------------------------
    # Store model and predictions
    # --------------------------------------------------------

    rf_v1_models[region] = model

    rf_v1_predictions[region] = {
        "y_true": y_validation.copy(),
        "y_pred": y_pred,
        "y_probability": y_probability
    }

    rf_v1_results.append({
        "REGION": region,
        "FORECAST_MODEL": "SARIMAX",
        "CLASSIFIER": "Random Forest v1",
        "PROBABILITY_THRESHOLD": 0.50,
        "TRUE_NEGATIVES": tn,
        "FALSE_POSITIVES": fp,
        "FALSE_NEGATIVES": fn,
        "TRUE_POSITIVES": tp,
        "PRECISION": precision,
        "RECALL": recall,
        "F1_SCORE": f1,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "FIT_TIME_SECONDS": fit_time_seconds
    })


rf_v1_results = pd.DataFrame(
    rf_v1_results
)

display(
    rf_v1_results.round(4)
)

,REGION,FORECAST_MODEL,CLASSIFIER,PROBABILITY_THRESHOLD,TRUE_NEGATIVES,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES,PRECISION,RECALL,F1_SCORE,ROC_AUC,PR_AUC,FIT_TIME_SECONDS
0,DOWNTOWN,SARIMAX,Random Forest v1,0.5000,4146,104,57,109,0.5117,0.6566,0.5752,0.9588,0.5474,0.6850
1,AIRPORT_WEST,SARIMAX,Random Forest v1,0.5000,3828,210,106,272,0.5643,0.7196,0.6326,0.9546,0.6493,0.4553


In [30]:
# ============================================================
# 16.2 RF V1 VS P97.5 THRESHOLD BASELINE
# ============================================================

rf_comparison = (
    baseline_results[
        [
            "REGION",
            "PRECISION",
            "RECALL",
            "F1_SCORE",
            "FALSE_POSITIVES",
            "FALSE_NEGATIVES",
            "TRUE_POSITIVES"
        ]
    ]
    .rename(
        columns={
            "PRECISION": "BASELINE_PRECISION",
            "RECALL": "BASELINE_RECALL",
            "F1_SCORE": "BASELINE_F1",
            "FALSE_POSITIVES": "BASELINE_FP",
            "FALSE_NEGATIVES": "BASELINE_FN",
            "TRUE_POSITIVES": "BASELINE_TP"
        }
    )
    .merge(
        rf_v1_results[
            [
                "REGION",
                "PRECISION",
                "RECALL",
                "F1_SCORE",
                "FALSE_POSITIVES",
                "FALSE_NEGATIVES",
                "TRUE_POSITIVES",
                "PR_AUC"
            ]
        ],
        on="REGION"
    )
    .rename(
        columns={
            "PRECISION": "RF_PRECISION",
            "RECALL": "RF_RECALL",
            "F1_SCORE": "RF_F1",
            "FALSE_POSITIVES": "RF_FP",
            "FALSE_NEGATIVES": "RF_FN",
            "TRUE_POSITIVES": "RF_TP",
            "PR_AUC": "RF_PR_AUC"
        }
    )
)

rf_comparison["F1_CHANGE"] = (
    rf_comparison["RF_F1"]
    - rf_comparison["BASELINE_F1"]
)

rf_comparison["RECALL_CHANGE"] = (
    rf_comparison["RF_RECALL"]
    - rf_comparison["BASELINE_RECALL"]
)

rf_comparison["FN_CHANGE"] = (
    rf_comparison["RF_FN"]
    - rf_comparison["BASELINE_FN"]
)

display(
    rf_comparison.round(4)
)

,REGION,BASELINE_PRECISION,BASELINE_RECALL,BASELINE_F1,BASELINE_FP,BASELINE_FN,BASELINE_TP,RF_PRECISION,RF_RECALL,RF_F1,RF_FP,RF_FN,RF_TP,RF_PR_AUC,F1_CHANGE,RECALL_CHANGE,FN_CHANGE
0,DOWNTOWN,0.6848,0.6807,0.6828,52,53,113,0.5117,0.6566,0.5752,104,57,109,0.5474,-0.1076,-0.0241,4
1,AIRPORT_WEST,0.7460,0.7381,0.7420,95,99,279,0.5643,0.7196,0.6326,210,106,272,0.6493,-0.1095,-0.0185,7


## 17. Random Forest Probability Threshold Analysis

The initial Random Forest classifier uses the conventional probability threshold of 0.50.

For an imbalanced Peak Risk classification problem, however, 0.50 is not necessarily the optimal operational decision threshold.

The validation period is therefore used to evaluate alternative probability thresholds.

Threshold selection is performed exclusively on validation data. The selected threshold is subsequently frozen before final evaluation on the 2025 test period.

The analysis evaluates the trade-off between:

- Precision
- Recall
- F1-score
- False Positives
- False Negatives

The objective is not to maximize Recall at any cost. A threshold that generates excessive false alarms may reduce the operational usefulness of the Peak Risk warning system.

In [31]:
# ============================================================
# 17.1 RANDOM FOREST PROBABILITY THRESHOLD SWEEP
# ============================================================

threshold_values = np.arange(
    0.05,
    0.951,
    0.01
)

rf_threshold_results = []

for region in REGIONS:

    y_true = (
        rf_v1_predictions[region]["y_true"]
    )

    y_probability = (
        rf_v1_predictions[region]["y_probability"]
    )

    for threshold in threshold_values:

        y_pred = (
            y_probability >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()

        precision = precision_score(
            y_true,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_true,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        rf_threshold_results.append({
            "REGION": region,
            "THRESHOLD": threshold,
            "PRECISION": precision,
            "RECALL": recall,
            "F1_SCORE": f1,
            "TRUE_NEGATIVES": tn,
            "FALSE_POSITIVES": fp,
            "FALSE_NEGATIVES": fn,
            "TRUE_POSITIVES": tp
        })

rf_threshold_results = pd.DataFrame(
    rf_threshold_results
)

print(
    "Threshold configurations evaluated:",
    len(rf_threshold_results)
)

Threshold configurations evaluated: 182


In [32]:
# ============================================================
# 17.2 BEST VALIDATION THRESHOLD BY F1
# ============================================================

best_rf_thresholds = (
    rf_threshold_results
    .sort_values(
        ["REGION", "F1_SCORE"],
        ascending=[True, False]
    )
    .groupby(
        "REGION",
        as_index=False
    )
    .first()
)

display(
    best_rf_thresholds.round(4)
)

,REGION,THRESHOLD,PRECISION,RECALL,F1_SCORE,TRUE_NEGATIVES,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES
0,AIRPORT_WEST,0.3600,0.5266,0.8122,0.6389,3762,276,71,307
1,DOWNTOWN,0.4000,0.4868,0.7771,0.5986,4114,136,37,129


In [33]:
# ============================================================
# 17.3 OPTIMIZED RF THRESHOLD VS P97.5 BASELINE
# ============================================================

optimized_threshold_comparison = (
    baseline_results[
        [
            "REGION",
            "PRECISION",
            "RECALL",
            "F1_SCORE",
            "FALSE_POSITIVES",
            "FALSE_NEGATIVES",
            "TRUE_POSITIVES"
        ]
    ]
    .rename(
        columns={
            "PRECISION": "BASELINE_PRECISION",
            "RECALL": "BASELINE_RECALL",
            "F1_SCORE": "BASELINE_F1",
            "FALSE_POSITIVES": "BASELINE_FP",
            "FALSE_NEGATIVES": "BASELINE_FN",
            "TRUE_POSITIVES": "BASELINE_TP"
        }
    )
    .merge(
        best_rf_thresholds[
            [
                "REGION",
                "THRESHOLD",
                "PRECISION",
                "RECALL",
                "F1_SCORE",
                "FALSE_POSITIVES",
                "FALSE_NEGATIVES",
                "TRUE_POSITIVES"
            ]
        ],
        on="REGION"
    )
    .rename(
        columns={
            "THRESHOLD": "RF_THRESHOLD",
            "PRECISION": "RF_PRECISION",
            "RECALL": "RF_RECALL",
            "F1_SCORE": "RF_F1",
            "FALSE_POSITIVES": "RF_FP",
            "FALSE_NEGATIVES": "RF_FN",
            "TRUE_POSITIVES": "RF_TP"
        }
    )
)

optimized_threshold_comparison[
    "F1_CHANGE"
] = (
    optimized_threshold_comparison["RF_F1"]
    - optimized_threshold_comparison["BASELINE_F1"]
)

optimized_threshold_comparison[
    "RECALL_CHANGE"
] = (
    optimized_threshold_comparison["RF_RECALL"]
    - optimized_threshold_comparison["BASELINE_RECALL"]
)

display(
    optimized_threshold_comparison.round(4)
)

,REGION,BASELINE_PRECISION,BASELINE_RECALL,BASELINE_F1,BASELINE_FP,BASELINE_FN,BASELINE_TP,RF_THRESHOLD,RF_PRECISION,RF_RECALL,RF_F1,RF_FP,RF_FN,RF_TP,F1_CHANGE,RECALL_CHANGE
0,DOWNTOWN,0.6848,0.6807,0.6828,52,53,113,0.4000,0.4868,0.7771,0.5986,136,37,129,-0.0842,0.0964
1,AIRPORT_WEST,0.7460,0.7381,0.7420,95,99,279,0.3600,0.5266,0.8122,0.6389,276,71,307,-0.1031,0.0741


## 18. Controlled Random Forest Hyperparameter Tuning

A controlled hyperparameter search is performed after the initial Random Forest and probability-threshold analyses.

The objective is to determine whether changes to Random Forest complexity and class-imbalance handling can improve Peak Risk detection without excessively increasing false alarms.

The temporal split remains unchanged:

- Training: January–June 2024
- Validation: July–December 2024

Random cross-validation is not used because it would violate the chronological structure of the forecasting problem.

Each candidate model is trained exclusively on the classifier-training period and evaluated on the subsequent validation period.

The search evaluates both model hyperparameters and classification probability thresholds.

The P97.5 threshold baseline remains the benchmark that the Random Forest must justify outperforming.

In [34]:
# ============================================================
# 18.1 CONTROLLED RANDOM FOREST HYPERPARAMETER SEARCH
# ============================================================

from itertools import product

rf_search_space = {
    "n_estimators": [300],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 3],
    "max_features": ["sqrt", 0.7],
    "class_weight": [
        "balanced",
        "balanced_subsample"
    ]
}

parameter_combinations = list(
    product(
        rf_search_space["n_estimators"],
        rf_search_space["max_depth"],
        rf_search_space["min_samples_leaf"],
        rf_search_space["max_features"],
        rf_search_space["class_weight"]
    )
)

print(
    "Model configurations per region:",
    len(parameter_combinations)
)

print(
    "Total model fits:",
    len(parameter_combinations) * len(REGIONS)
)

Model configurations per region: 24
Total model fits: 48


In [36]:
# ============================================================
# 18.2 TRAIN CANDIDATES AND OPTIMIZE VALIDATION THRESHOLD
# ============================================================

rf_tuning_results = []
rf_tuned_models = {}

for region in REGIONS:

    print("=" * 60)
    print("TUNING:", region)

    data = rf_datasets[region]

    X_train = data["X_train"]
    y_train = data["y_train"]

    X_validation = data["X_validation"]
    y_validation = data["y_validation"]

    region_models = {}

    for config_id, params in enumerate(
        parameter_combinations,
        start=1
    ):

        (
            n_estimators,
            max_depth,
            min_samples_leaf,
            max_features,
            class_weight
        ) = params

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=2,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        y_probability = model.predict_proba(
            X_validation
        )[:, 1]

        pr_auc = average_precision_score(
            y_validation,
            y_probability
        )

        roc_auc = roc_auc_score(
            y_validation,
            y_probability
        )

        # Find best threshold for this model
        best_candidate = None

        for threshold in threshold_values:

            y_pred = (
                y_probability >= threshold
            ).astype(int)

            tn, fp, fn, tp = confusion_matrix(
                y_validation,
                y_pred,
                labels=[0, 1]
            ).ravel()

            precision = precision_score(
                y_validation,
                y_pred,
                zero_division=0
            )

            recall = recall_score(
                y_validation,
                y_pred,
                zero_division=0
            )

            f1 = f1_score(
                y_validation,
                y_pred,
                zero_division=0
            )

            candidate = {
                "REGION": region,
                "CONFIG_ID": config_id,
                "N_ESTIMATORS": n_estimators,
                "MAX_DEPTH": max_depth,
                "MIN_SAMPLES_LEAF": min_samples_leaf,
                "MAX_FEATURES": max_features,
                "CLASS_WEIGHT": class_weight,
                "THRESHOLD": threshold,
                "PRECISION": precision,
                "RECALL": recall,
                "F1_SCORE": f1,
                "PR_AUC": pr_auc,
                "ROC_AUC": roc_auc,
                "TRUE_NEGATIVES": tn,
                "FALSE_POSITIVES": fp,
                "FALSE_NEGATIVES": fn,
                "TRUE_POSITIVES": tp
            }

            if (
                best_candidate is None
                or f1 > best_candidate["F1_SCORE"]
            ):
                best_candidate = candidate

        rf_tuning_results.append(
            best_candidate
        )

        region_models[config_id] = model

    rf_tuned_models[region] = region_models


rf_tuning_results = pd.DataFrame(
    rf_tuning_results
)

print(
    "\nCompleted configurations:",
    len(rf_tuning_results)
)

TUNING: DOWNTOWN
TUNING: AIRPORT_WEST

Completed configurations: 48


In [37]:
# ============================================================
# 18.3 BEST TUNED RANDOM FOREST BY REGION
# ============================================================

best_tuned_rf = (
    rf_tuning_results
    .sort_values(
        [
            "REGION",
            "F1_SCORE",
            "RECALL",
            "PR_AUC"
        ],
        ascending=[
            True,
            False,
            False,
            False
        ]
    )
    .groupby(
        "REGION",
        as_index=False
    )
    .first()
)

display(
    best_tuned_rf.round(4)
)

,REGION,CONFIG_ID,N_ESTIMATORS,MAX_DEPTH,MIN_SAMPLES_LEAF,MAX_FEATURES,CLASS_WEIGHT,THRESHOLD,PRECISION,RECALL,F1_SCORE,PR_AUC,ROC_AUC,TRUE_NEGATIVES,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES
0,AIRPORT_WEST,22,300,12.0000,3,sqrt,balanced_subsample,0.5300,0.6310,0.7328,0.6781,0.6806,0.9622,3876,162,101,277
1,DOWNTOWN,6,300,8.0000,3,sqrt,balanced_subsample,0.5000,0.5517,0.7711,0.6432,0.5894,0.9586,4146,104,38,128


In [38]:
# ============================================================
# 18.4 TUNED RANDOM FOREST VS P97.5 BASELINE
# ============================================================

tuned_rf_comparison = (
    baseline_results[
        [
            "REGION",
            "PRECISION",
            "RECALL",
            "F1_SCORE",
            "FALSE_POSITIVES",
            "FALSE_NEGATIVES",
            "TRUE_POSITIVES"
        ]
    ]
    .rename(
        columns={
            "PRECISION": "BASELINE_PRECISION",
            "RECALL": "BASELINE_RECALL",
            "F1_SCORE": "BASELINE_F1",
            "FALSE_POSITIVES": "BASELINE_FP",
            "FALSE_NEGATIVES": "BASELINE_FN",
            "TRUE_POSITIVES": "BASELINE_TP"
        }
    )
    .merge(
        best_tuned_rf[
            [
                "REGION",
                "CONFIG_ID",
                "THRESHOLD",
                "PRECISION",
                "RECALL",
                "F1_SCORE",
                "PR_AUC",
                "FALSE_POSITIVES",
                "FALSE_NEGATIVES",
                "TRUE_POSITIVES"
            ]
        ],
        on="REGION"
    )
    .rename(
        columns={
            "THRESHOLD": "RF_THRESHOLD",
            "PRECISION": "RF_PRECISION",
            "RECALL": "RF_RECALL",
            "F1_SCORE": "RF_F1",
            "PR_AUC": "RF_PR_AUC",
            "FALSE_POSITIVES": "RF_FP",
            "FALSE_NEGATIVES": "RF_FN",
            "TRUE_POSITIVES": "RF_TP"
        }
    )
)

tuned_rf_comparison["F1_CHANGE"] = (
    tuned_rf_comparison["RF_F1"]
    - tuned_rf_comparison["BASELINE_F1"]
)

tuned_rf_comparison["RECALL_CHANGE"] = (
    tuned_rf_comparison["RF_RECALL"]
    - tuned_rf_comparison["BASELINE_RECALL"]
)

tuned_rf_comparison["FN_CHANGE"] = (
    tuned_rf_comparison["RF_FN"]
    - tuned_rf_comparison["BASELINE_FN"]
)

display(
    tuned_rf_comparison.round(4)
)

,REGION,BASELINE_PRECISION,BASELINE_RECALL,BASELINE_F1,BASELINE_FP,BASELINE_FN,BASELINE_TP,CONFIG_ID,RF_THRESHOLD,RF_PRECISION,RF_RECALL,RF_F1,RF_PR_AUC,RF_FP,RF_FN,RF_TP,F1_CHANGE,RECALL_CHANGE,FN_CHANGE
0,DOWNTOWN,0.6848,0.6807,0.6828,52,53,113,6,0.5000,0.5517,0.7711,0.6432,0.5894,104,38,128,-0.0396,0.0904,-15
1,AIRPORT_WEST,0.7460,0.7381,0.7420,95,99,279,22,0.5300,0.6310,0.7328,0.6781,0.6806,162,101,277,-0.0639,-0.0053,2


### Validation Interpretation

Controlled Random Forest tuning improved classification performance relative to the initial Random Forest configuration.

However, the tuned Random Forest did not outperform the direct P97.5 threshold baseline in F1-score for either region.

For Downtown, the tuned classifier increased Recall but generated substantially more false positives, resulting in a lower overall F1-score than the threshold baseline.

For Airport-West, the tuned classifier remained below the threshold baseline in both F1-score and Recall.

Therefore, the current evidence does not support replacing the simple forecast-threshold Peak Risk rule with the Random Forest classifier solely on the basis of validation performance.

The Random Forest remains useful as an alternative risk-ranking approach because its probability outputs provide continuous risk scores and may support different operational trade-offs between missed peaks and false alarms.

Final conclusions regarding deployment should be based on the untouched 2025 test period rather than validation performance alone.

## 19. Freeze Selected Random Forest Configuration

Random Forest model selection is now complete.

All hyperparameters and probability thresholds were selected exclusively using the July–December 2024 validation period.

The selected configurations are frozen before accessing the 2025 test forecasts:

- Downtown: Random Forest configuration 6, probability threshold 0.50
- Airport-West: Random Forest configuration 22, probability threshold 0.53

No further model, feature, hyperparameter, or probability-threshold selection will be performed using the 2025 test period.

The 2025 dataset will therefore serve exclusively as the final out-of-sample evaluation period.

In [39]:
# ============================================================
# 19.1 FREEZE SELECTED RANDOM FOREST CONFIGURATIONS
# ============================================================

selected_rf_configurations = {}

for _, row in best_tuned_rf.iterrows():

    region = row["REGION"]

    selected_rf_configurations[region] = {
        "config_id": int(row["CONFIG_ID"]),
        "n_estimators": int(row["N_ESTIMATORS"]),
        "max_depth": (
            None
            if pd.isna(row["MAX_DEPTH"])
            else int(row["MAX_DEPTH"])
        ),
        "min_samples_leaf": int(row["MIN_SAMPLES_LEAF"]),
        "max_features": row["MAX_FEATURES"],
        "class_weight": row["CLASS_WEIGHT"],
        "probability_threshold": float(row["THRESHOLD"])
    }


for region, config in selected_rf_configurations.items():

    print("=" * 60)
    print(region)

    for key, value in config.items():
        print(f"{key}: {value}")

AIRPORT_WEST
config_id: 22
n_estimators: 300
max_depth: 12
min_samples_leaf: 3
max_features: sqrt
class_weight: balanced_subsample
probability_threshold: 0.5300000000000001
DOWNTOWN
config_id: 6
n_estimators: 300
max_depth: 8
min_samples_leaf: 3
max_features: sqrt
class_weight: balanced_subsample
probability_threshold: 0.5000000000000001


In [40]:
# ============================================================
# 20.1 COMBINE JAN–JUN + JUL–DEC 2024 FORECASTS
# ============================================================

sarimax_final_classifier_train = pd.concat(
    [
        sarimax_classifier_train_standardized,
        sarimax_validation_standardized
    ],
    ignore_index=True
)

sarimax_final_classifier_train = (
    sarimax_final_classifier_train
    .sort_values(
        ["REGION", "TIMESTAMP"]
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("FINAL SARIMAX CLASSIFIER TRAINING FORECASTS — 2024")
print("=" * 60)

print(
    "Shape:",
    sarimax_final_classifier_train.shape
)

print("\nRows by region:")
print(
    sarimax_final_classifier_train[
        "REGION"
    ].value_counts()
)

print(
    "\nPeriod:",
    sarimax_final_classifier_train["TIMESTAMP"].min(),
    "to",
    sarimax_final_classifier_train["TIMESTAMP"].max()
)

print(
    "\nDuplicate REGION/TIMESTAMP:",
    sarimax_final_classifier_train.duplicated(
        subset=["REGION", "TIMESTAMP"]
    ).sum()
)

print(
    "Missing predicted consumption:",
    sarimax_final_classifier_train[
        "PREDICTED_CONSUMPTION"
    ].isna().sum()
)

FINAL SARIMAX CLASSIFIER TRAINING FORECASTS — 2024
Shape: (17568, 7)

Rows by region:
REGION
AIRPORT_WEST    8784
DOWNTOWN        8784
Name: count, dtype: int64

Period: 2024-01-01 00:00:00 to 2024-12-31 23:00:00

Duplicate REGION/TIMESTAMP: 0
Missing predicted consumption: 0


In [41]:
# ============================================================
# 20.2 VALIDATE FINAL 2024 FORECAST CONTRACT
# ============================================================

for region in REGIONS:

    region_forecasts = (
        sarimax_final_classifier_train[
            sarimax_final_classifier_train["REGION"] == region
        ]
        .copy()
    )

    validate_forecast_contract(
        region_forecasts
    )

Forecast contract validation: PASSED
Rows: 8784
Models: ['SARIMAX']
Regions: ['DOWNTOWN']
Period: 2024-01-01 00:00:00 to 2024-12-31 23:00:00
Forecast contract validation: PASSED
Rows: 8784
Models: ['SARIMAX']
Regions: ['AIRPORT_WEST']
Period: 2024-01-01 00:00:00 to 2024-12-31 23:00:00


In [42]:
# ============================================================
# 20.3 BUILD FINAL 2024 CLASSIFIER FEATURES
# ============================================================

sarimax_final_classifier_peak = (
    add_actual_peak_risk_target(
        sarimax_final_classifier_train
    )
)

sarimax_final_classifier_features = (
    build_peak_risk_features(
        sarimax_final_classifier_peak
    )
)

print("=" * 60)
print("FINAL RANDOM FOREST TRAINING DATASET — 2024")
print("=" * 60)

print(
    "Rows:",
    len(sarimax_final_classifier_features)
)

print(
    "Missing RF feature values:",
    sarimax_final_classifier_features[
        RF_FEATURES
    ].isna().sum().sum()
)

final_training_distribution = (
    sarimax_final_classifier_features
    .groupby("REGION")[
        "ACTUAL_PEAK_RISK"
    ]
    .agg(
        TOTAL_HOURS="count",
        PEAK_HOURS="sum"
    )
)

final_training_distribution[
    "PEAK_RATE_PERCENT"
] = (
    final_training_distribution["PEAK_HOURS"]
    /
    final_training_distribution["TOTAL_HOURS"]
    * 100
)

display(
    final_training_distribution.round(2)
)

FINAL RANDOM FOREST TRAINING DATASET — 2024
Rows: 17568
Missing RF feature values: 0


,TOTAL_HOURS,PEAK_HOURS,PEAK_RATE_PERCENT
REGION,,,
AIRPORT_WEST,8784,466,5.3100
DOWNTOWN,8784,253,2.8800


## 21. Final Random Forest Refit

After model selection was completed using the July–December 2024 validation period, the selected Random Forest configurations are refitted using all available out-of-sample 2024 forecasts.

The final training period therefore covers January 1 through December 31, 2024.

The previously selected hyperparameters and probability thresholds remain frozen.

No model-selection decisions are made during this refit.

The resulting classifiers will be evaluated once on the untouched 2025 test forecasts.

In [43]:
# ============================================================
# 21.1 REFIT FINAL RANDOM FOREST MODELS ON FULL 2024
# ============================================================

final_rf_models = {}

for region in REGIONS:

    print("=" * 60)
    print(region)

    region_data = (
        sarimax_final_classifier_features[
            sarimax_final_classifier_features["REGION"] == region
        ]
        .sort_values("TIMESTAMP")
        .copy()
    )

    X_final_train = (
        region_data[RF_FEATURES]
        .copy()
    )

    y_final_train = (
        region_data["ACTUAL_PEAK_RISK"]
        .copy()
    )

    config = selected_rf_configurations[region]

    final_model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_split=2,
        min_samples_leaf=config["min_samples_leaf"],
        max_features=config["max_features"],
        class_weight=config["class_weight"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    start_time = time.time()

    final_model.fit(
        X_final_train,
        y_final_train
    )

    fit_time = time.time() - start_time

    final_rf_models[region] = final_model

    print(
        "Training shape:",
        X_final_train.shape
    )

    print(
        "Peak observations:",
        int(y_final_train.sum())
    )

    print(
        "Peak rate:",
        f"{y_final_train.mean() * 100:.2f}%"
    )

    print(
        "Frozen threshold:",
        round(
            config["probability_threshold"],
            2
        )
    )

    print(
        "Fit time:",
        f"{fit_time:.2f} seconds"
    )

DOWNTOWN
Training shape: (8784, 9)
Peak observations: 253
Peak rate: 2.88%
Frozen threshold: 0.5
Fit time: 0.72 seconds
AIRPORT_WEST
Training shape: (8784, 9)
Peak observations: 466
Peak rate: 5.31%
Frozen threshold: 0.53
Fit time: 0.72 seconds


## 22. Final 2025 Test Forecast Preparation

The final Random Forest classifiers are evaluated using the untouched 2025 SARIMAX forecasts.

These forecasts cover January 1 through December 31, 2025 and were not used for:

- Random Forest training;
- hyperparameter selection;
- probability-threshold selection.

The 2025 forecasts are standardized using the same forecast contract and transformed using exactly the same feature-engineering procedure used during classifier development.

No model parameters or decision thresholds are modified based on 2025 results.

In [44]:
# ============================================================
# 22.2 LOAD FINAL SARIMAX 2025 TEST FORECASTS
# ============================================================

SARIMAX_FINAL_TEST_FILES = {
    "DOWNTOWN": (
        MODEL_OUTPUT_PATH /
        "downtown_final_c3_calendar_weather_2025_test_predictions.csv"
    ),
    "AIRPORT_WEST": (
        MODEL_OUTPUT_PATH /
        "airport_west_final_c3_calendar_weather_2025_test_predictions.csv"
    )
}

sarimax_raw_final_test = {}

for region, file_path in SARIMAX_FINAL_TEST_FILES.items():

    df = pd.read_csv(
        file_path,
        parse_dates=["TIMESTAMP"]
    )

    sarimax_raw_final_test[region] = df

    print("=" * 60)
    print(region)
    print("File:", file_path.name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    print(
        "Period:",
        df["TIMESTAMP"].min(),
        "to",
        df["TIMESTAMP"].max()
    )

DOWNTOWN
File: downtown_final_c3_calendar_weather_2025_test_predictions.csv
Shape: (8760, 3)
Columns: ['TIMESTAMP', 'ACTUAL', 'PREDICTED']
Period: 2025-01-01 00:00:00 to 2025-12-31 23:00:00
AIRPORT_WEST
File: airport_west_final_c3_calendar_weather_2025_test_predictions.csv
Shape: (8760, 3)
Columns: ['TIMESTAMP', 'ACTUAL', 'PREDICTED']
Period: 2025-01-01 00:00:00 to 2025-12-31 23:00:00


In [45]:
# ============================================================
# 22.3 STANDARDIZE FINAL SARIMAX 2025 TEST FORECASTS
# ============================================================

sarimax_final_test_standardized = []

for region in REGIONS:

    standardized = adapt_sarimax_rolling_forecast(
        df=sarimax_raw_final_test[region],
        region=region,
        horizon=24
    )

    sarimax_final_test_standardized.append(
        standardized
    )

sarimax_final_test_standardized = pd.concat(
    sarimax_final_test_standardized,
    ignore_index=True
)

print("=" * 60)
print("FINAL SARIMAX 2025 TEST FORECASTS")
print("=" * 60)

print(
    "Shape:",
    sarimax_final_test_standardized.shape
)

print("\nRows by region:")
print(
    sarimax_final_test_standardized[
        "REGION"
    ].value_counts()
)

print(
    "\nPeriod:",
    sarimax_final_test_standardized["TIMESTAMP"].min(),
    "to",
    sarimax_final_test_standardized["TIMESTAMP"].max()
)

print(
    "\nDuplicate REGION/TIMESTAMP:",
    sarimax_final_test_standardized.duplicated(
        subset=["REGION", "TIMESTAMP"]
    ).sum()
)

Forecast contract validation: PASSED
Rows: 8760
Models: ['SARIMAX']
Regions: ['DOWNTOWN']
Period: 2025-01-01 00:00:00 to 2025-12-31 23:00:00
Forecast contract validation: PASSED
Rows: 8760
Models: ['SARIMAX']
Regions: ['AIRPORT_WEST']
Period: 2025-01-01 00:00:00 to 2025-12-31 23:00:00
FINAL SARIMAX 2025 TEST FORECASTS
Shape: (17520, 7)

Rows by region:
REGION
DOWNTOWN        8760
AIRPORT_WEST    8760
Name: count, dtype: int64

Period: 2025-01-01 00:00:00 to 2025-12-31 23:00:00

Duplicate REGION/TIMESTAMP: 0


In [47]:
# ============================================================
# 22.4 BUILD FINAL 2025 TEST TARGET AND FEATURES
# ============================================================

sarimax_final_test_peak = (
    add_actual_peak_risk_target(
        sarimax_final_test_standardized
    )
)

sarimax_final_test_features = (
    build_peak_risk_features(
        sarimax_final_test_peak
    )
)

print("=" * 60)
print("FINAL RANDOM FOREST TEST DATASET — 2025")
print("=" * 60)

print(
    "Rows:",
    len(sarimax_final_test_features)
)

print(
    "Missing RF feature values:",
    sarimax_final_test_features[
        RF_FEATURES
    ].isna().sum().sum()
)

test_distribution = (
    sarimax_final_test_features
    .groupby("REGION")[
        "ACTUAL_PEAK_RISK"
    ]
    .agg(
        TOTAL_HOURS="count",
        PEAK_HOURS="sum"
    )
)

test_distribution[
    "PEAK_RATE_PERCENT"
] = (
    test_distribution["PEAK_HOURS"]
    /
    test_distribution["TOTAL_HOURS"]
    * 100
)

display(
    test_distribution.round(2)
)

FINAL RANDOM FOREST TEST DATASET — 2025
Rows: 17520
Missing RF feature values: 0


,TOTAL_HOURS,PEAK_HOURS,PEAK_RATE_PERCENT
REGION,,,
AIRPORT_WEST,8760,586,6.6900
DOWNTOWN,8760,754,8.6100


## 23. Final 2025 Test Evaluation

The frozen Random Forest Peak Risk classifiers are now evaluated once on the untouched 2025 test period.

The final models were refitted using all available 2024 out-of-sample forecasting observations while preserving the hyperparameters and probability thresholds selected during validation.

No information from the 2025 test period is used to modify the classifier.

Performance is compared against the direct P97.5 forecast-threshold baseline using the same observations and Peak Risk definition.

The final evaluation considers:

- Precision
- Recall
- F1-score
- PR-AUC
- ROC-AUC
- False Positives
- False Negatives
- True Positives

This comparison determines whether the Random Forest layer provides additional classification value beyond the simpler forecast-threshold rule.

In [48]:
# ============================================================
# 23.2 FINAL RANDOM FOREST EVALUATION — 2025 TEST
# ============================================================

final_rf_test_results = []
final_rf_test_predictions = {}

for region in REGIONS:

    region_test = (
        sarimax_final_test_features[
            sarimax_final_test_features["REGION"] == region
        ]
        .sort_values("TIMESTAMP")
        .copy()
    )

    X_test = (
        region_test[RF_FEATURES]
        .copy()
    )

    y_test = (
        region_test["ACTUAL_PEAK_RISK"]
        .copy()
    )

    model = final_rf_models[region]

    threshold = (
        selected_rf_configurations[region][
            "probability_threshold"
        ]
    )

    # --------------------------------------------------------
    # Frozen model predictions
    # --------------------------------------------------------

    y_probability = model.predict_proba(
        X_test
    )[:, 1]

    y_pred = (
        y_probability >= threshold
    ).astype(int)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_probability
    )

    pr_auc = average_precision_score(
        y_test,
        y_probability
    )

    # --------------------------------------------------------
    # Store predictions
    # --------------------------------------------------------

    prediction_output = region_test[
        [
            "REGION",
            "TIMESTAMP",
            "MODEL_NAME",
            "ACTUAL_CONSUMPTION",
            "PREDICTED_CONSUMPTION",
            "PEAK_THRESHOLD",
            "ACTUAL_PEAK_RISK"
        ]
    ].copy()

    prediction_output[
        "RF_PEAK_PROBABILITY"
    ] = y_probability

    prediction_output[
        "RF_PEAK_PREDICTION"
    ] = y_pred

    final_rf_test_predictions[
        region
    ] = prediction_output

    # --------------------------------------------------------
    # Store metrics
    # --------------------------------------------------------

    final_rf_test_results.append({
        "REGION": region,
        "FORECAST_MODEL": "SARIMAX",
        "CLASSIFIER": "Random Forest",
        "THRESHOLD": threshold,
        "TEST_ROWS": len(y_test),
        "ACTUAL_PEAKS": int(y_test.sum()),
        "TRUE_NEGATIVES": tn,
        "FALSE_POSITIVES": fp,
        "FALSE_NEGATIVES": fn,
        "TRUE_POSITIVES": tp,
        "PRECISION": precision,
        "RECALL": recall,
        "F1_SCORE": f1,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc
    })


final_rf_test_results = pd.DataFrame(
    final_rf_test_results
)

display(
    final_rf_test_results.round(4)
)

,REGION,FORECAST_MODEL,CLASSIFIER,THRESHOLD,TEST_ROWS,ACTUAL_PEAKS,TRUE_NEGATIVES,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES,PRECISION,RECALL,F1_SCORE,ROC_AUC,PR_AUC
0,DOWNTOWN,SARIMAX,Random Forest,0.5000,8760,754,7719,287,89,665,0.6985,0.8820,0.7796,0.9824,0.8179
1,AIRPORT_WEST,SARIMAX,Random Forest,0.5300,8760,586,8000,174,44,542,0.7570,0.9249,0.8326,0.9893,0.8706


In [49]:
# ============================================================
# 23.3 FINAL P97.5 BASELINE — SAME 2025 TEST OBSERVATIONS
# ============================================================

final_baseline_test_results = []

for region in REGIONS:

    region_test = (
        sarimax_final_test_features[
            sarimax_final_test_features["REGION"] == region
        ]
        .sort_values("TIMESTAMP")
        .copy()
    )

    y_test = (
        region_test["ACTUAL_PEAK_RISK"]
        .copy()
    )

    baseline_pred = (
        region_test["PREDICTED_CONSUMPTION"]
        >=
        region_test["PEAK_THRESHOLD"]
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        baseline_pred,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_test,
        baseline_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        baseline_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        baseline_pred,
        zero_division=0
    )

    final_baseline_test_results.append({
        "REGION": region,
        "FORECAST_MODEL": "SARIMAX",
        "CLASSIFIER": "P97.5 Threshold Baseline",
        "TEST_ROWS": len(y_test),
        "ACTUAL_PEAKS": int(y_test.sum()),
        "TRUE_NEGATIVES": tn,
        "FALSE_POSITIVES": fp,
        "FALSE_NEGATIVES": fn,
        "TRUE_POSITIVES": tp,
        "PRECISION": precision,
        "RECALL": recall,
        "F1_SCORE": f1
    })


final_baseline_test_results = pd.DataFrame(
    final_baseline_test_results
)

display(
    final_baseline_test_results.round(4)
)

,REGION,FORECAST_MODEL,CLASSIFIER,TEST_ROWS,ACTUAL_PEAKS,TRUE_NEGATIVES,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES,PRECISION,RECALL,F1_SCORE
0,DOWNTOWN,SARIMAX,P97.5 Threshold Baseline,8760,754,7869,137,148,606,0.8156,0.8037,0.8096
1,AIRPORT_WEST,SARIMAX,P97.5 Threshold Baseline,8760,586,8068,106,107,479,0.8188,0.8174,0.8181


In [50]:
# ============================================================
# 24.1 FINAL TEST COMPARISON — RF VS P97.5 BASELINE
# ============================================================

final_test_comparison = (
    final_baseline_test_results[
        [
            "REGION",
            "PRECISION",
            "RECALL",
            "F1_SCORE",
            "FALSE_POSITIVES",
            "FALSE_NEGATIVES",
            "TRUE_POSITIVES"
        ]
    ]
    .rename(
        columns={
            "PRECISION": "BASELINE_PRECISION",
            "RECALL": "BASELINE_RECALL",
            "F1_SCORE": "BASELINE_F1",
            "FALSE_POSITIVES": "BASELINE_FP",
            "FALSE_NEGATIVES": "BASELINE_FN",
            "TRUE_POSITIVES": "BASELINE_TP"
        }
    )
    .merge(
        final_rf_test_results[
            [
                "REGION",
                "THRESHOLD",
                "PRECISION",
                "RECALL",
                "F1_SCORE",
                "ROC_AUC",
                "PR_AUC",
                "FALSE_POSITIVES",
                "FALSE_NEGATIVES",
                "TRUE_POSITIVES"
            ]
        ],
        on="REGION"
    )
    .rename(
        columns={
            "THRESHOLD": "RF_THRESHOLD",
            "PRECISION": "RF_PRECISION",
            "RECALL": "RF_RECALL",
            "F1_SCORE": "RF_F1",
            "ROC_AUC": "RF_ROC_AUC",
            "PR_AUC": "RF_PR_AUC",
            "FALSE_POSITIVES": "RF_FP",
            "FALSE_NEGATIVES": "RF_FN",
            "TRUE_POSITIVES": "RF_TP"
        }
    )
)

final_test_comparison["F1_CHANGE"] = (
    final_test_comparison["RF_F1"]
    - final_test_comparison["BASELINE_F1"]
)

final_test_comparison["RECALL_CHANGE"] = (
    final_test_comparison["RF_RECALL"]
    - final_test_comparison["BASELINE_RECALL"]
)

final_test_comparison["PRECISION_CHANGE"] = (
    final_test_comparison["RF_PRECISION"]
    - final_test_comparison["BASELINE_PRECISION"]
)

final_test_comparison["FN_REDUCTION"] = (
    final_test_comparison["BASELINE_FN"]
    - final_test_comparison["RF_FN"]
)

final_test_comparison["ADDITIONAL_FP"] = (
    final_test_comparison["RF_FP"]
    - final_test_comparison["BASELINE_FP"]
)

display(
    final_test_comparison.round(4)
)

,REGION,BASELINE_PRECISION,BASELINE_RECALL,BASELINE_F1,BASELINE_FP,BASELINE_FN,BASELINE_TP,RF_THRESHOLD,RF_PRECISION,RF_RECALL,RF_F1,RF_ROC_AUC,RF_PR_AUC,RF_FP,RF_FN,RF_TP,F1_CHANGE,RECALL_CHANGE,PRECISION_CHANGE,FN_REDUCTION,ADDITIONAL_FP
0,DOWNTOWN,0.8156,0.8037,0.8096,137,148,606,0.5000,0.6985,0.8820,0.7796,0.9824,0.8179,287,89,665,-0.0300,0.0782,-0.1171,59,150
1,AIRPORT_WEST,0.8188,0.8174,0.8181,106,107,479,0.5300,0.7570,0.9249,0.8326,0.9893,0.8706,174,44,542,0.0145,0.1075,-0.0618,63,68


## 25. Save SARIMAX Peak Risk Classification Results

The completed SARIMAX Peak Risk experiment is preserved before extending the reusable classification pipeline to additional forecasting models.

The saved artifacts include:

- final 2025 Random Forest Peak Risk predictions;
- final Random Forest test metrics;
- P97.5 threshold baseline metrics;
- direct Random Forest versus baseline comparison.

These outputs preserve the untouched 2025 evaluation results and allow subsequent forecasting models to be evaluated independently using the same classification architecture.

In [51]:
# ============================================================
# 25.1 SAVE FINAL SARIMAX RANDOM FOREST TEST PREDICTIONS
# ============================================================

for region in REGIONS:

    output_file = (
        PEAK_RISK_OUTPUT_PATH /
        f"{region.lower()}_sarimax_random_forest_peak_risk_2025_test_predictions.csv"
    )

    final_rf_test_predictions[
        region
    ].to_csv(
        output_file,
        index=False
    )

    print(
        f"{region}: saved -> {output_file.name}"
    )

DOWNTOWN: saved -> downtown_sarimax_random_forest_peak_risk_2025_test_predictions.csv
AIRPORT_WEST: saved -> airport_west_sarimax_random_forest_peak_risk_2025_test_predictions.csv


In [52]:
# ============================================================
# 25.2 SAVE FINAL SARIMAX PEAK RISK METRICS
# ============================================================

rf_metrics_file = (
    PEAK_RISK_OUTPUT_PATH /
    "sarimax_random_forest_peak_risk_2025_test_metrics.csv"
)

baseline_metrics_file = (
    PEAK_RISK_OUTPUT_PATH /
    "sarimax_p975_peak_risk_2025_test_metrics.csv"
)

comparison_file = (
    PEAK_RISK_OUTPUT_PATH /
    "sarimax_peak_risk_2025_final_comparison.csv"
)

final_rf_test_results.to_csv(
    rf_metrics_file,
    index=False
)

final_baseline_test_results.to_csv(
    baseline_metrics_file,
    index=False
)

final_test_comparison.to_csv(
    comparison_file,
    index=False
)

print("Saved:")
print("-", rf_metrics_file.name)
print("-", baseline_metrics_file.name)
print("-", comparison_file.name)

Saved:
- sarimax_random_forest_peak_risk_2025_test_metrics.csv
- sarimax_p975_peak_risk_2025_test_metrics.csv
- sarimax_peak_risk_2025_final_comparison.csv


In [53]:
# ============================================================
# 25.3 VERIFY SAVED SARIMAX PEAK RISK OUTPUTS
# ============================================================

saved_peak_risk_files = sorted(
    PEAK_RISK_OUTPUT_PATH.glob(
        "*sarimax*"
    )
)

print(
    "SARIMAX Peak Risk files found:",
    len(saved_peak_risk_files)
)

for file in saved_peak_risk_files:
    print("-", file.name)

SARIMAX Peak Risk files found: 5
- airport_west_sarimax_random_forest_peak_risk_2025_test_predictions.csv
- downtown_sarimax_random_forest_peak_risk_2025_test_predictions.csv
- sarimax_p975_peak_risk_2025_test_metrics.csv
- sarimax_peak_risk_2025_final_comparison.csv
- sarimax_random_forest_peak_risk_2025_test_metrics.csv


## 26. Reusable Multi-Model Peak Risk Pipeline

The SARIMAX experiment established and validated the complete Peak Risk classification workflow.

The next step is to generalize this workflow so that the same classification architecture can be applied independently to the remaining forecasting models:

- Seasonal Naïve
- XGBoost Regressor
- LightGBM Regressor
- Random Forest Regressor

Each forecasting model will be processed independently using:

1. the standardized forecast contract;
2. the same Peak Risk target definition;
3. the same feature-engineering procedure;
4. the same chronological classifier train-validation-test framework;
5. controlled Random Forest tuning using validation data only;
6. frozen final configurations before 2025 test evaluation;
7. direct comparison against the P97.5 forecast-threshold baseline.

The classification architecture is reusable across forecasting models, while classifier parameters and probability thresholds are selected independently for each forecasting-model and region combination.

In [54]:
# ============================================================
# 26.1 GENERIC FORECAST ADAPTER
# ============================================================

def adapt_generic_rolling_forecast(
    df,
    region,
    model_name,
    timestamp_column="TIMESTAMP",
    actual_column="ACTUAL",
    predicted_column="PREDICTED",
    horizon=24
):
    """
    Convert a generic rolling 24-hour forecasting output into
    the standardized Peak Risk forecast contract.

    Assumes sequential forecast blocks with horizons 1..24.
    """

    if model_name not in FORECAST_MODELS:
        raise ValueError(
            f"Unsupported forecasting model: {model_name}"
        )

    data = (
        df.copy()
        .sort_values(timestamp_column)
        .reset_index(drop=True)
    )

    required_native_columns = [
        timestamp_column,
        actual_column,
        predicted_column
    ]

    missing = [
        col for col in required_native_columns
        if col not in data.columns
    ]

    if missing:
        raise ValueError(
            f"Forecast output missing columns: {missing}"
        )

    data[timestamp_column] = pd.to_datetime(
        data[timestamp_column]
    )

    data["REGION"] = region
    data["MODEL_NAME"] = model_name

    data["FORECAST_HORIZON_HOURS"] = (
        np.arange(len(data)) % horizon
    ) + 1

    data["FORECAST_ORIGIN"] = (
        data[timestamp_column]
        - pd.to_timedelta(
            data["FORECAST_HORIZON_HOURS"],
            unit="h"
        )
    )

    data = data.rename(
        columns={
            timestamp_column: "TIMESTAMP",
            actual_column: "ACTUAL_CONSUMPTION",
            predicted_column: "PREDICTED_CONSUMPTION"
        }
    )

    data = data[
        REQUIRED_FORECAST_COLUMNS
    ].copy()

    return validate_forecast_contract(
        data
    )

## 27. Current Multi-Model Integration Status

The Peak Risk classification pipeline has been fully implemented and validated using SARIMAX forecast outputs.

The architecture is designed to support the remaining forecasting models:

- Seasonal Naïve
- XGBoost Regressor
- LightGBM Regressor
- Random Forest Regressor

However, their standardized train, validation, and test forecast outputs are not yet available.

Therefore, empirical Random Forest Peak Risk evaluation has currently been completed only for SARIMAX.

Once the remaining forecasting outputs become available, each model will be processed independently through the same standardized pipeline without changing the Peak Risk target definition, temporal evaluation framework, or classification methodology.

## 28. Random Forest Feature Importance

Feature importance is examined for the final Random Forest classifiers to identify which forecasting and temporal variables contribute most strongly to Peak Risk classification.

The analysis uses the final models refitted on the complete 2024 development period.

Feature importance describes how strongly each variable contributes to Random Forest decision-making. It does not establish causal relationships between the variables and electricity peak events.

Because separate classifiers are fitted for Downtown and Airport-West, feature importance is evaluated independently for each region.

In [55]:
# ============================================================
# 28.1 FINAL RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

feature_importance_results = []

for region in REGIONS:

    model = final_rf_models[region]

    importance_df = pd.DataFrame({
        "REGION": region,
        "FEATURE": RF_FEATURES,
        "IMPORTANCE": model.feature_importances_
    })

    importance_df = (
        importance_df
        .sort_values(
            "IMPORTANCE",
            ascending=False
        )
        .reset_index(drop=True)
    )

    importance_df["RANK"] = (
        np.arange(1, len(importance_df) + 1)
    )

    feature_importance_results.append(
        importance_df
    )


feature_importance_results = pd.concat(
    feature_importance_results,
    ignore_index=True
)

for region in REGIONS:

    print("=" * 60)
    print(region)

    display(
        feature_importance_results[
            feature_importance_results["REGION"] == region
        ][
            [
                "RANK",
                "FEATURE",
                "IMPORTANCE"
            ]
        ].round(4)
    )

DOWNTOWN


,RANK,FEATURE,IMPORTANCE
0,1,PREDICTED_CONSUMPTION,0.3125
1,2,PREDICTED_LOAD_RATIO,0.2722
2,3,DISTANCE_TO_THRESHOLD,0.2558
3,4,HOUR,0.0514
4,5,FORECAST_HORIZON_HOURS,0.0474
5,6,MONTH,0.0410
6,7,DAY_OF_WEEK,0.0166
7,8,IS_WEEKEND,0.0031
8,9,PEAK_THRESHOLD,0.0000


AIRPORT_WEST


,RANK,FEATURE,IMPORTANCE
9,1,PREDICTED_CONSUMPTION,0.3085
10,2,PREDICTED_LOAD_RATIO,0.2645
11,3,DISTANCE_TO_THRESHOLD,0.2635
12,4,MONTH,0.0612
13,5,HOUR,0.0443
14,6,FORECAST_HORIZON_HOURS,0.0385
15,7,DAY_OF_WEEK,0.0161
16,8,IS_WEEKEND,0.0035
17,9,PEAK_THRESHOLD,0.0000


In [56]:
# ============================================================
# 28.2 FEATURE IMPORTANCE SANITY CHECK
# ============================================================

importance_check = (
    feature_importance_results
    .groupby("REGION")["IMPORTANCE"]
    .sum()
)

print("Feature importance sum by region:")
print(
    importance_check.round(6)
)

Feature importance sum by region:
REGION
AIRPORT_WEST   1.0000
DOWNTOWN       1.0000
Name: IMPORTANCE, dtype: float64


## 29. Final 2025 Classification Error Analysis

Confusion matrices are used to examine the operational behavior of the final Peak Risk classifiers on the untouched 2025 test period.

Particular attention is given to False Negatives because they represent actual Peak Risk hours that the warning system failed to identify.

Random Forest results are compared directly with the P97.5 forecast-threshold baseline using the same observations.

In [57]:
# ============================================================
# 29.1 FINAL 2025 CLASSIFICATION ERROR SUMMARY
# ============================================================

classification_error_summary = []

for region in REGIONS:

    rf_row = final_rf_test_results[
        final_rf_test_results["REGION"] == region
    ].iloc[0]

    baseline_row = final_baseline_test_results[
        final_baseline_test_results["REGION"] == region
    ].iloc[0]

    for method, row in [
        ("P97.5 Threshold Baseline", baseline_row),
        ("Random Forest", rf_row)
    ]:

        classification_error_summary.append({
            "REGION": region,
            "METHOD": method,
            "TRUE_NEGATIVES": int(row["TRUE_NEGATIVES"]),
            "FALSE_POSITIVES": int(row["FALSE_POSITIVES"]),
            "FALSE_NEGATIVES": int(row["FALSE_NEGATIVES"]),
            "TRUE_POSITIVES": int(row["TRUE_POSITIVES"]),
            "PRECISION": row["PRECISION"],
            "RECALL": row["RECALL"],
            "F1_SCORE": row["F1_SCORE"]
        })


classification_error_summary = pd.DataFrame(
    classification_error_summary
)

display(
    classification_error_summary.round(4)
)

,REGION,METHOD,TRUE_NEGATIVES,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES,PRECISION,RECALL,F1_SCORE
0,DOWNTOWN,P97.5 Threshold Baseline,7869,137,148,606,0.8156,0.8037,0.8096
1,DOWNTOWN,Random Forest,7719,287,89,665,0.6985,0.8820,0.7796
2,AIRPORT_WEST,P97.5 Threshold Baseline,8068,106,107,479,0.8188,0.8174,0.8181
3,AIRPORT_WEST,Random Forest,8000,174,44,542,0.7570,0.9249,0.8326


In [58]:
# ============================================================
# 29.2 RANDOM FOREST ERROR TRADE-OFF
# ============================================================

rf_error_tradeoff = (
    final_test_comparison[
        [
            "REGION",
            "BASELINE_FN",
            "RF_FN",
            "FN_REDUCTION",
            "BASELINE_FP",
            "RF_FP",
            "ADDITIONAL_FP"
        ]
    ]
    .copy()
)

rf_error_tradeoff["FN_REDUCTION_PERCENT"] = (
    rf_error_tradeoff["FN_REDUCTION"]
    /
    rf_error_tradeoff["BASELINE_FN"]
    * 100
)

rf_error_tradeoff["ADDITIONAL_FP_PER_FN_AVOIDED"] = (
    rf_error_tradeoff["ADDITIONAL_FP"]
    /
    rf_error_tradeoff["FN_REDUCTION"]
)

display(
    rf_error_tradeoff.round(2)
)

,REGION,BASELINE_FN,RF_FN,FN_REDUCTION,BASELINE_FP,RF_FP,ADDITIONAL_FP,FN_REDUCTION_PERCENT,ADDITIONAL_FP_PER_FN_AVOIDED
0,DOWNTOWN,148,89,59,137,287,150,39.8600,2.5400
1,AIRPORT_WEST,107,44,63,106,174,68,58.8800,1.0800


## 30. False Negative Analysis

False Negatives represent actual Peak Risk hours that the Random Forest classifier failed to identify.

Because missed peak events may represent the most operationally significant classification error, the remaining False Negatives are examined to determine whether they share systematic characteristics.

The analysis focuses on:

- actual and predicted electricity consumption;
- distance of the forecast from the Peak Risk threshold;
- Random Forest probability;
- forecast horizon;
- hour of day;
- month.

This analysis is diagnostic only. No model parameters or probability thresholds are modified based on the 2025 test results.

In [59]:
# ============================================================
# 30.1 IDENTIFY FINAL RANDOM FOREST FALSE NEGATIVES
# ============================================================

final_rf_false_negatives = {}

for region in REGIONS:

    predictions = (
        final_rf_test_predictions[region]
        .copy()
    )

    false_negatives = predictions[
        (predictions["ACTUAL_PEAK_RISK"] == 1)
        &
        (predictions["RF_PEAK_PREDICTION"] == 0)
    ].copy()

    final_rf_false_negatives[region] = (
        false_negatives
    )

    print("=" * 60)
    print(region)

    print(
        "False Negatives:",
        len(false_negatives)
    )

    display(
        false_negatives.head()
    )

DOWNTOWN
False Negatives: 89


,REGION,TIMESTAMP,MODEL_NAME,ACTUAL_CONSUMPTION,PREDICTED_CONSUMPTION,PEAK_THRESHOLD,ACTUAL_PEAK_RISK,RF_PEAK_PROBABILITY,RF_PEAK_PREDICTION
141,DOWNTOWN,2025-01-06 21:00:00,SARIMAX,"35,215.3000","33,715.2099","34,974.1100",1,0.2823,0
357,DOWNTOWN,2025-01-15 21:00:00,SARIMAX,"35,142.1000","34,947.5490","34,974.1100",1,0.4721,0
452,DOWNTOWN,2025-01-19 20:00:00,SARIMAX,"36,572.0000","33,575.9200","34,974.1100",1,0.2089,0
453,DOWNTOWN,2025-01-19 21:00:00,SARIMAX,"35,016.9000","32,302.5729","34,974.1100",1,0.0495,0
783,DOWNTOWN,2025-02-02 15:00:00,SARIMAX,"34,998.6000","32,589.0918","34,974.1100",1,0.2943,0


AIRPORT_WEST
False Negatives: 44


,REGION,TIMESTAMP,MODEL_NAME,ACTUAL_CONSUMPTION,PREDICTED_CONSUMPTION,PEAK_THRESHOLD,ACTUAL_PEAK_RISK,RF_PEAK_PROBABILITY,RF_PEAK_PREDICTION
12805,AIRPORT_WEST,2025-06-18 13:00:00,SARIMAX,"50,069.8000","44,861.0707","48,865.8500",1,0.4655,0
12876,AIRPORT_WEST,2025-06-21 12:00:00,SARIMAX,"49,159.6000","40,721.2048","48,865.8500",1,0.0146,0
12896,AIRPORT_WEST,2025-06-22 08:00:00,SARIMAX,"49,703.5000","41,035.8454","48,865.8500",1,0.0000,0
12897,AIRPORT_WEST,2025-06-22 09:00:00,SARIMAX,"57,286.8000","45,203.8276","48,865.8500",1,0.1529,0
12910,AIRPORT_WEST,2025-06-22 22:00:00,SARIMAX,"63,233.3000","44,089.6541","48,865.8500",1,0.1013,0


In [60]:
# ============================================================
# 30.2 FALSE NEGATIVE DIAGNOSTIC FEATURES
# ============================================================

for region in REGIONS:

    fn = final_rf_false_negatives[region].copy()

    fn["FORECAST_ERROR"] = (
        fn["PREDICTED_CONSUMPTION"]
        - fn["ACTUAL_CONSUMPTION"]
    )

    fn["ABS_FORECAST_ERROR"] = (
        fn["FORECAST_ERROR"].abs()
    )

    fn["FORECAST_ERROR_PERCENT"] = (
        fn["FORECAST_ERROR"]
        /
        fn["ACTUAL_CONSUMPTION"]
        * 100
    )

    fn["PREDICTED_DISTANCE_TO_THRESHOLD"] = (
        fn["PREDICTED_CONSUMPTION"]
        - fn["PEAK_THRESHOLD"]
    )

    fn["ACTUAL_DISTANCE_TO_THRESHOLD"] = (
        fn["ACTUAL_CONSUMPTION"]
        - fn["PEAK_THRESHOLD"]
    )

    fn["HOUR"] = (
        pd.to_datetime(fn["TIMESTAMP"]).dt.hour
    )

    fn["MONTH"] = (
        pd.to_datetime(fn["TIMESTAMP"]).dt.month
    )

    final_rf_false_negatives[region] = fn

In [61]:
# ============================================================
# 30.3 FALSE NEGATIVE DIAGNOSTIC SUMMARY
# ============================================================

fn_diagnostic_summary = []

for region in REGIONS:

    fn = final_rf_false_negatives[region]

    threshold = (
        selected_rf_configurations[region][
            "probability_threshold"
        ]
    )

    fn_diagnostic_summary.append({
        "REGION": region,
        "FALSE_NEGATIVES": len(fn),

        "AVG_ACTUAL_CONSUMPTION":
            fn["ACTUAL_CONSUMPTION"].mean(),

        "AVG_PREDICTED_CONSUMPTION":
            fn["PREDICTED_CONSUMPTION"].mean(),

        "AVG_FORECAST_ERROR":
            fn["FORECAST_ERROR"].mean(),

        "AVG_FORECAST_ERROR_PERCENT":
            fn["FORECAST_ERROR_PERCENT"].mean(),

        "AVG_ACTUAL_DISTANCE_TO_THRESHOLD":
            fn["ACTUAL_DISTANCE_TO_THRESHOLD"].mean(),

        "AVG_PREDICTED_DISTANCE_TO_THRESHOLD":
            fn["PREDICTED_DISTANCE_TO_THRESHOLD"].mean(),

        "AVG_RF_PROBABILITY":
            fn["RF_PEAK_PROBABILITY"].mean(),

        "DECISION_THRESHOLD":
            threshold
    })


fn_diagnostic_summary = pd.DataFrame(
    fn_diagnostic_summary
)

display(
    fn_diagnostic_summary.round(4)
)

,REGION,FALSE_NEGATIVES,AVG_ACTUAL_CONSUMPTION,AVG_PREDICTED_CONSUMPTION,AVG_FORECAST_ERROR,AVG_FORECAST_ERROR_PERCENT,AVG_ACTUAL_DISTANCE_TO_THRESHOLD,AVG_PREDICTED_DISTANCE_TO_THRESHOLD,AVG_RF_PROBABILITY,DECISION_THRESHOLD
0,DOWNTOWN,89,"37,159.6135","32,563.4841","-4,596.1294",-11.7648,"2,185.5035","-2,410.6259",0.2213,0.5000
1,AIRPORT_WEST,44,"51,885.1523","45,005.6460","-6,879.5062",-13.0217,"3,019.3023","-3,860.2040",0.2783,0.5300
